# REPATH 01 -- Generate the 50-feature candidate dataset

This notebook reads the UITestFix site folders under `data/`, reproduces the benchmark query set, constructs candidate shortlists, and exports the ordered 50-feature representation used for training.

The site groups are loaded from `configs/site_split.yaml`. Large generated feature files and diagnostics are written under `generated/repath/features/`. The query manifest required by calibration is written to `artifacts/repath/repath_query_manifest.csv`.

Run this notebook before the training notebook.


In [1]:
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional
import csv
import json
import re

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from repath_evaluation.data import load_site_split


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "configs" / "site_split.yaml").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from the repository root or the notebooks directory."
    )


print("Imports ready.")


Imports ready.


## 1. Configure repository paths and site groups


In [2]:
REPO_ROOT = find_repository_root()
DATASET_ROOT = REPO_ROOT / "data"
FEATURE_OUTPUT_DIR = REPO_ROOT / "generated" / "repath" / "features"
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "repath"
SPLIT_CONFIG = REPO_ROOT / "configs" / "site_split.yaml"

FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

QUERY_CSV = FEATURE_OUTPUT_DIR / "repath_exact_framework_queries.csv"
WORKING_PAIR_CSV = FEATURE_OUTPUT_DIR / "pairs_real_query_shortlist_presemantic.csv"
PAIR_CSV = FEATURE_OUTPUT_DIR / "pairs_real_query_shortlist_50_features.csv"
QUERY_MANIFEST_CSV = ARTIFACT_ROOT / "repath_query_manifest.csv"
FEATURE_SCHEMA_CSV = FEATURE_OUTPUT_DIR / "repath_50_feature_schema.csv"
SBERT_CACHE = FEATURE_OUTPUT_DIR / "repath_sbert_embedding_cache.pkl"
SITE_REPORT_CSV = FEATURE_OUTPUT_DIR / "repath_query_shortlist_site_report.csv"
QUERY_REPORT_CSV = FEATURE_OUTPUT_DIR / "repath_query_shortlist_query_report.csv"
ISSUES_CSV = FEATURE_OUTPUT_DIR / "repath_query_shortlist_issues.csv"
MISSING_POSITIVE_AUDIT_CSV = (
    FEATURE_OUTPUT_DIR / "repath_query_shortlist_missing_positive_diagnostics.csv"
)
FEATURE_AUDIT_CSV = FEATURE_OUTPUT_DIR / "repath_query_shortlist_feature_audit.csv"

SBERT_MODEL_NAME = "all-MiniLM-L6-v2"
SBERT_BATCH_SIZE = 128
KEEP_INTERMEDIATE_PAIR_CSV = False

MAX_CANDIDATES = 1000
MIN_ACCESSIBLE_TEXT_LENGTH = 2
MAX_ACCESSIBLE_TEXT_LENGTH = 150
INTERACTIVE_ANCESTOR_MAX_DEPTH = 1
MAX_INTERACTIVE_DESCENDANT_CHILDREN = 0
FALLBACK_TOP_K = 25
FALLBACK_MIN_SCORE = 0.15

USE_GENERIC_ACCESSIBLE_TEXT = True
USE_ATTRIBUTE_INTERACTION = True
USE_INTERACTIVE_DESCENDANTS = True
USE_QUERY_SPECIFIC_FALLBACK = True

QUERY_SHORTLIST_TOP_K = 25

PROTECT_EXACT_ID = True
PROTECT_EXACT_DATA_TEST_ID = True
PROTECT_EXACT_NAME = True
PROTECT_EXACT_ACCESSIBLE_TEXT = True
FORCE_INCLUDE_POSITIVE = False

site_split = load_site_split(SPLIT_CONFIG)
DEVELOPMENT_SITES = set(site_split.development_sites)
TEST_SITES = set(site_split.test_sites)
ALL_SITES = DEVELOPMENT_SITES | TEST_SITES

assert len(DEVELOPMENT_SITES) == 16
assert len(TEST_SITES) == 6
assert len(ALL_SITES) == 22
assert not (DEVELOPMENT_SITES & TEST_SITES)


def split_for_site(site: str) -> str:
    if site in DEVELOPMENT_SITES:
        return "development"
    if site in TEST_SITES:
        return "test"
    raise KeyError(f"No split configured for {site!r}")


print("Repository root       :", REPO_ROOT)
print("Dataset root          :", DATASET_ROOT)
print("Generated features    :", FEATURE_OUTPUT_DIR)
print("Query manifest        :", QUERY_MANIFEST_CSV)
print("SBERT model           :", SBERT_MODEL_NAME)
print("Query shortlist top-K :", QUERY_SHORTLIST_TOP_K)
print("Development sites     :", len(DEVELOPMENT_SITES))
print("Held-out test sites   :", len(TEST_SITES))


Repository root       : /workspaces/REPATH-Evaluation-Suite
Dataset root          : /workspaces/REPATH-Evaluation-Suite/data
Generated features    : /workspaces/REPATH-Evaluation-Suite/generated/repath/features
Query manifest        : /workspaces/REPATH-Evaluation-Suite/artifacts/repath/repath_query_manifest.csv
SBERT model           : all-MiniLM-L6-v2
Query shortlist top-K : 25
Development sites     : 16
Held-out test sites   : 6


## 2. Load UITestFix files


In [3]:
CONTAINER = 1  # frame.config.Settings.CONTAINER

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8-sig") as handle:
        return json.load(handle)

def safe_int(value: Any, default: Optional[int] = None) -> Optional[int]:
    try:
        if value is None or str(value).strip() == "":
            return default
        return int(float(str(value).strip()))
    except (TypeError, ValueError):
        return default

def find_site_dir(root: Path, configured_name: str) -> Optional[Path]:
    direct = root / configured_name
    if direct.exists():
        return direct

    if not root.exists():
        return None

    by_lower = {
        path.name.lower(): path
        for path in root.iterdir()
        if path.is_dir()
    }
    return by_lower.get(configured_name.lower())

def require_dataset_file(site_dir: Path, relative_path: str) -> Path:
    exact = site_dir / relative_path
    if exact.exists():
        return exact

    # Case-insensitive fallback for Windows/Linux differences.
    parts = Path(relative_path).parts
    current = site_dir
    for part in parts:
        if not current.exists() or not current.is_dir():
            break
        children = {
            child.name.lower(): child
            for child in current.iterdir()
        }
        match = children.get(part.lower())
        if match is None:
            break
        current = match
    else:
        if current.exists():
            return current

    raise FileNotFoundError(
        f"Required dataset file not found: {site_dir / relative_path}"
    )

def read_match_map(result_path: Path) -> Dict[int, int]:
    data = load_json(result_path)

    if not isinstance(data, dict):
        raise ValueError(f"{result_path} does not contain a JSON object.")

    match_map = data.get("matchMap")
    if not isinstance(match_map, dict):
        raise ValueError(f"No dictionary-valued matchMap in {result_path}.")

    output = {}
    for old_value, new_value in match_map.items():
        old_index = safe_int(old_value)
        new_index = safe_int(new_value)
        if old_index is None or new_index is None:
            raise ValueError(
                f"Non-integer matchMap entry: {old_value!r} -> {new_value!r}"
            )
        output[old_index] = new_index

    return output

def read_dom_list(dom_path: Path) -> List[Dict[str, Any]]:
    data = load_json(dom_path)
    if not isinstance(data, list):
        raise ValueError(
            f"{dom_path} must be a JSON array because JsonProcess "
            "deserialises it as List<DomNodeInfo>."
        )
    if not all(isinstance(node, dict) for node in data):
        raise ValueError(f"{dom_path} contains a non-object list item.")
    return data

print("Exact dataset readers ready.")


Exact dataset readers ready.


## 3. Build the benchmark query set


In [4]:
query_rows = []
site_report_rows = []
issue_rows = []

for site in sorted(ALL_SITES, key=str.lower):
    split = split_for_site(site)
    site_dir = find_site_dir(DATASET_ROOT, site)

    if site_dir is None:
        site_report_rows.append({
            "site": site,
            "split": split,
            "matchmap_entries": 0,
            "container_entries_skipped": 0,
            "framework_queries": 0,
            "status": "site_folder_not_found",
        })
        issue_rows.append({
            "site": site,
            "split": split,
            "old_idx": "",
            "true_new_idx": "",
            "issue": "site_folder_not_found",
        })
        continue

    try:
        result_path = require_dataset_file(
            site_dir,
            "result/result.json",
        )
        old_dom_path = require_dataset_file(
            site_dir,
            "old/domNodeInfo.json",
        )
        new_dom_path = require_dataset_file(
            site_dir,
            "new/domNodeInfo.json",
        )

        match_map = read_match_map(result_path)
        old_nodes = read_dom_list(old_dom_path)
        new_nodes = read_dom_list(new_dom_path)
    except Exception as error:
        site_report_rows.append({
            "site": site,
            "split": split,
            "matchmap_entries": 0,
            "container_entries_skipped": 0,
            "framework_queries": 0,
            "status": f"load_error: {error}",
        })
        issue_rows.append({
            "site": site,
            "split": split,
            "old_idx": "",
            "true_new_idx": "",
            "issue": f"load_error: {error}",
        })
        continue

    skipped_containers = 0
    accepted_queries = 0

    for old_index, true_new_index in match_map.items():
        # Main.report uses oldDomNodeInfoList.get(index).
        if old_index < 0 or old_index >= len(old_nodes):
            issue_rows.append({
                "site": site,
                "split": split,
                "old_idx": old_index,
                "true_new_idx": true_new_index,
                "issue": "old_matchmap_index_out_of_range",
            })
            continue

        # Needed later to create a positive candidate row.
        if true_new_index < 0 or true_new_index >= len(new_nodes):
            issue_rows.append({
                "site": site,
                "split": split,
                "old_idx": old_index,
                "true_new_idx": true_new_index,
                "issue": "true_new_matchmap_index_out_of_range",
            })
            continue

        old_node = old_nodes[old_index]
        old_element_type = safe_int(
            old_node.get("elementType"),
            CONTAINER,
        )

        # Exact Main.report(...) condition.
        if old_element_type == CONTAINER:
            skipped_containers += 1
            continue

        query_rows.append({
            "qid": f"{site}::{old_index}",
            "site": site,
            "split": split,
            "old_idx": old_index,
            "true_new_idx": true_new_index,
            "old_element_type": old_element_type,
            "old_is_leaf": bool(old_node.get("isLeaf", False)),
            "old_is_displayed": bool(old_node.get("isDisplayed", False)),
            "old_tag": str(old_node.get("tagName", "") or ""),
            "old_text": str(old_node.get("text", "") or ""),
            "result_json": str(result_path),
            "old_dom_json": str(old_dom_path),
            "new_dom_json": str(new_dom_path),
        })
        accepted_queries += 1

    site_report_rows.append({
        "site": site,
        "split": split,
        "matchmap_entries": len(match_map),
        "container_entries_skipped": skipped_containers,
        "framework_queries": accepted_queries,
        "status": "ok",
    })

queries_df = pd.DataFrame(query_rows)
site_report_df = pd.DataFrame(site_report_rows)
issues_df = pd.DataFrame(issue_rows)

queries_df.to_csv(QUERY_CSV, index=False)
site_report_df.to_csv(SITE_REPORT_CSV, index=False)
issues_df.to_csv(ISSUES_CSV, index=False)

display(site_report_df)
display(queries_df.head(20))

print("Exact framework queries:", len(queries_df))
print("Unique qids            :", queries_df["qid"].nunique())
print("Query CSV              :", QUERY_CSV)

assert len(queries_df) == queries_df["qid"].nunique()
assert not queries_df.duplicated(["site", "old_idx"]).any()

# Exact accounting for sites loaded successfully.
loaded = site_report_df[site_report_df["status"] == "ok"]
assert (
    loaded["matchmap_entries"]
    == loaded["container_entries_skipped"] + loaded["framework_queries"]
).all()

print("Exact WebUIExperimentFramework query extraction passed.")


,site,split,matchmap_entries,container_entries_skipped,framework_queries,status
0,amazon-2017-2020,development,80,4,76,ok
1,apple-2018-2020,development,64,2,62,ok
2,beijing-2017-2019,test,110,7,103,ok
3,Bing,development,13,2,11,ok
4,book-2016-2019,test,115,8,107,ok
5,facebook-2020-2022,development,44,0,44,ok
6,fandom,development,110,27,83,ok
7,foodnetwork-2018-2019,development,27,0,27,ok
8,google,development,11,0,11,ok
9,instagram,test,13,1,12,ok


,qid,site,split,old_idx,true_new_idx,old_element_type,old_is_leaf,old_is_displayed,old_tag,old_text,result_json,old_dom_json,new_dom_json
0,amazon-2017-2020::7,amazon-2017-2020,development,7,8,3,True,True,a,Amazon,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
1,amazon-2017-2020::11,amazon-2017-2020,development,11,26,3,True,True,div,All All Departments Alexa Skills Amazon Device...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
2,amazon-2017-2020::13,amazon-2017-2020,development,13,29,3,True,True,span,Go,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
3,amazon-2017-2020::14,amazon-2017-2020,development,14,30,3,True,True,input,Go,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
4,amazon-2017-2020::19,amazon-2017-2020,development,19,35,3,True,True,a,Sign in,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
5,amazon-2017-2020::20,amazon-2017-2020,development,20,36,3,True,True,div,New customer? Start here.,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
6,amazon-2017-2020::30,amazon-2017-2020,development,30,14,3,True,True,span,Hello. Sign in,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
7,amazon-2017-2020::32,amazon-2017-2020,development,32,15,3,True,True,a,Orders,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
8,amazon-2017-2020::33,amazon-2017-2020,development,33,18,3,True,True,a,Try Prime,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...
9,amazon-2017-2020::36,amazon-2017-2020,development,36,22,3,True,True,span,Cart,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...,/workspaces/REPATH-Evaluation-Suite/data/amazo...


Exact framework queries: 1100
Unique qids            : 1100
Query CSV              : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_exact_framework_queries.csv
Exact WebUIExperimentFramework query extraction passed.


## 4. Define candidate retrieval and shortlist helpers

The page-wide pool uses native/attribute-based interactive elements, generic accessible-text leaves, direct leaf descendants of interactive elements, and a small query-specific fallback. A second cheap retrieval score then shortlists that pool independently for each old target.


In [5]:
def normalise_text(value: Any) -> str:
    text = "" if value is None else str(value).strip().lower()
    return re.sub(r"\s+", " ", text)


def normalise_tokens(value: Any) -> str:
    text = normalise_text(value)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def token_set(value: Any) -> set:
    return {token for token in normalise_tokens(value).split() if token}


def token_jaccard(left: Any, right: Any) -> float:
    left_tokens, right_tokens = token_set(left), token_set(right)
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


def sequence_similarity(left: Any, right: Any) -> float:
    left_text, right_text = normalise_text(left), normalise_text(right)
    if not left_text and not right_text:
        return 1.0
    if not left_text or not right_text:
        return 0.0
    return float(SequenceMatcher(None, left_text, right_text).ratio())


def node_attributes(node: Dict[str, Any]) -> Dict[str, str]:
    raw = node.get("attributes", {})
    if not isinstance(raw, dict):
        raw = {}
    return {
        str(key).strip().lower(): str(value)
        for key, value in raw.items() if value is not None
    }


def node_attribute(node: Dict[str, Any], *names: str) -> str:
    attributes = node_attributes(node)
    for name in names:
        value = attributes.get(name.lower())
        if value is not None and str(value) != "":
            return str(value)

    aliases = {
        "aria-label": ["ariaLabel", "arialabel"],
        "data-testid": ["dataTestId", "datatestid"],
        "data-test": ["dataTest", "datatest"],
        "data-qa": ["dataQa", "dataqa"],
        "class": ["className", "classname"],
        "tabindex": ["tabIndex"],
    }
    for name in names:
        keys = [name, name.lower(), name.replace("-", "")] + aliases.get(name.lower(), [])
        for key in keys:
            value = node.get(key)
            if value is not None and str(value) != "":
                return str(value)
    return ""


def first_number(node: Dict[str, Any], names: Iterable[str], default: float = 0.0) -> float:
    for name in names:
        value = node.get(name)
        try:
            if value is not None and str(value).strip() != "":
                return float(value)
        except (TypeError, ValueError):
            pass
    return float(default)


def first_bool(node: Dict[str, Any], names: Iterable[str], default: bool) -> bool:
    for name in names:
        if name not in node or node.get(name) is None:
            continue
        value = node.get(name)
        if isinstance(value, bool):
            return value
        text = str(value).strip().lower()
        if text in {"true", "1", "yes"}:
            return True
        if text in {"false", "0", "no"}:
            return False
    return default


def node_tag(node: Dict[str, Any]) -> str:
    return normalise_text(
        node.get("tagName", node.get("tag", node.get("name", "")))
    )


def raw_node_text(node: Dict[str, Any]) -> str:
    for key in ["text", "innerText", "textContent", "value"]:
        value = normalise_text(node.get(key, ""))
        if value:
            return value
    return ""


def accessible_text(node: Dict[str, Any]) -> str:
    """First meaningful user-facing text source, independent of tag name."""
    values = [
        raw_node_text(node),
        node_attribute(node, "aria-label", "arialabel"),
        node_attribute(node, "title"),
        node_attribute(node, "alt"),
        node_attribute(node, "value"),
        node_attribute(node, "placeholder"),
        node_attribute(node, "labeltext", "labelText"),
    ]
    for value in values:
        value = normalise_text(value)
        if value:
            return value
    return ""


def node_text(node: Dict[str, Any]) -> str:
    # Rich combined text used by the pair features and fallback scoring.
    values = [
        raw_node_text(node),
        node_attribute(node, "labeltext", "labelText"),
        node_attribute(node, "placeholder"),
        node_attribute(node, "aria-label", "arialabel"),
        node_attribute(node, "title"),
        node_attribute(node, "alt"),
        node_attribute(node, "name"),
        node_attribute(node, "id"),
        node_attribute(node, "data-testid", "data-test", "data-qa"),
    ]
    return normalise_text(" ".join(value for value in values if value))


def node_xpath(node: Dict[str, Any]) -> str:
    for key in ["xpath", "xPath", "XPath", "absoluteXPath", "fullXPath"]:
        value = normalise_text(node.get(key, ""))
        if value:
            return value
    return normalise_text(node_attribute(node, "xpath"))


def dataset_visible(node: Dict[str, Any]) -> bool:
    # Offline approximation of runtime display/visibility/opacity and bbox >= 2x2.
    displayed = first_bool(
        node,
        ["isDisplayed", "displayed", "isVisible", "visible"],
        default=True,
    )
    width = first_number(node, ["width", "bboxW", "w"], 0.0)
    height = first_number(node, ["height", "bboxH", "h"], 0.0)
    return displayed and width >= 2.0 and height >= 2.0


def element_child_count(node: Dict[str, Any]) -> Optional[int]:
    for key in ["children", "childNodes", "childElementIds"]:
        value = node.get(key)
        if isinstance(value, list):
            return len(value)
    for key in ["childrenCount", "childCount", "numberOfChildren"]:
        value = safe_int(node.get(key))
        if value is not None:
            return value
    if "isLeaf" in node:
        return 0 if first_bool(node, ["isLeaf"], default=False) else None
    return None


def has_no_element_children(node: Dict[str, Any]) -> bool:
    child_count = element_child_count(node)
    if child_count is not None:
        return child_count == 0
    # elementType is a useful dataset fallback, but not exact live DOM state.
    return safe_int(node.get("elementType"), CONTAINER) != CONTAINER


INTERACTIVE_TAGS = {"input", "textarea", "select", "button"}
INTERACTIVE_ROLES = {
    "button", "link", "menuitem", "menuitemcheckbox", "menuitemradio",
    "tab", "checkbox", "radio", "textbox", "searchbox", "combobox",
    "switch", "option", "slider", "spinbutton", "treeitem",
}


def valid_href(value: Any) -> bool:
    href = normalise_text(value)
    return bool(href) and href != "#" and not href.startswith("javascript")


def parse_tabindex(node: Dict[str, Any]) -> Optional[int]:
    value = node_attribute(node, "tabindex")
    return safe_int(value, None)


def is_interactive_candidate(node: Dict[str, Any]) -> bool:
    if not dataset_visible(node):
        return False

    tag = node_tag(node)
    role = normalise_text(node_attribute(node, "role"))
    if tag in INTERACTIVE_TAGS:
        return True

    if not USE_ATTRIBUTE_INTERACTION:
        return tag == "a" and valid_href(node_attribute(node, "href"))

    tabindex = parse_tabindex(node)
    return (
        valid_href(node_attribute(node, "href"))
        or role in INTERACTIVE_ROLES
        or bool(node_attribute(node, "onclick"))
        or normalise_text(node_attribute(node, "contenteditable")) == "true"
        or (tabindex is not None and tabindex >= 0)
    )


def is_generic_text_leaf_candidate(node: Dict[str, Any]) -> bool:
    if not USE_GENERIC_ACCESSIBLE_TEXT or not dataset_visible(node):
        return False
    if is_interactive_candidate(node):
        return False
    if not has_no_element_children(node):
        return False
    text = accessible_text(node)
    return MIN_ACCESSIBLE_TEXT_LENGTH <= len(text) <= MAX_ACCESSIBLE_TEXT_LENGTH


def _node_reference_map(nodes: List[Dict[str, Any]]) -> Dict[str, int]:
    mapping = {}
    for index, node in enumerate(nodes):
        for key in ["nodeId", "nodeID", "backendNodeId", "backendNodeID", "uid"]:
            value = node.get(key)
            if value is not None and str(value).strip() != "":
                mapping.setdefault(str(value).strip(), index)
    return mapping


def _parent_xpath(xpath: str) -> str:
    xpath = normalise_text(xpath).rstrip("/")
    if not xpath or "/" not in xpath:
        return ""
    return xpath.rsplit("/", 1)[0]


def build_parent_index_map(nodes: List[Dict[str, Any]]) -> Dict[int, Optional[int]]:
    """Resolve parent links from explicit indices/IDs, then fall back to XPath."""
    parent_map: Dict[int, Optional[int]] = {index: None for index in range(len(nodes))}
    reference_map = _node_reference_map(nodes)

    xpath_to_index = {}
    for index, node in enumerate(nodes):
        xpath = node_xpath(node)
        if xpath:
            xpath_to_index.setdefault(xpath, index)

    index_keys = [
        "parentIndex", "parentIdx", "parent_index", "parentNodeIndex",
        "parentElementIndex",
    ]
    reference_keys = [
        "parentNodeId", "parentNodeID", "parentBackendNodeId",
        "parentBackendNodeID", "parentId",
    ]

    for index, node in enumerate(nodes):
        resolved = None

        for key in index_keys:
            candidate = safe_int(node.get(key), None)
            if candidate is not None and 0 <= candidate < len(nodes) and candidate != index:
                resolved = candidate
                break

        if resolved is None:
            for key in reference_keys:
                value = node.get(key)
                if value is None:
                    continue
                candidate = reference_map.get(str(value).strip())
                if candidate is not None and candidate != index:
                    resolved = candidate
                    break

        if resolved is None:
            generic_parent = node.get("parent")
            if isinstance(generic_parent, dict):
                for key in ["index", "idx", "nodeIndex", "nodeId", "backendNodeId"]:
                    if key not in generic_parent:
                        continue
                    value = generic_parent.get(key)
                    candidate = safe_int(value, None)
                    if key.lower().endswith("index") or key in {"index", "idx"}:
                        if candidate is not None and 0 <= candidate < len(nodes) and candidate != index:
                            resolved = candidate
                            break
                    candidate = reference_map.get(str(value).strip())
                    if candidate is not None and candidate != index:
                        resolved = candidate
                        break
            elif generic_parent is not None:
                candidate = safe_int(generic_parent, None)
                if candidate is not None and 0 <= candidate < len(nodes) and candidate != index:
                    resolved = candidate

        if resolved is None:
            parent_xpath = _parent_xpath(node_xpath(node))
            candidate = xpath_to_index.get(parent_xpath)
            if candidate is not None and candidate != index:
                resolved = candidate

        parent_map[index] = resolved

    return parent_map


def nearest_interactive_ancestor(
    node_index: int,
    nodes: List[Dict[str, Any]],
    parent_map: Dict[int, Optional[int]],
    max_depth: int = INTERACTIVE_ANCESTOR_MAX_DEPTH,
):
    current = parent_map.get(node_index)
    for depth in range(1, max_depth + 1):
        if current is None or current < 0 or current >= len(nodes):
            break
        if is_interactive_candidate(nodes[current]):
            return current, depth
        current = parent_map.get(current)
    return None, None


def is_interactive_descendant_candidate(
    node_index: int,
    nodes: List[Dict[str, Any]],
    parent_map: Dict[int, Optional[int]],
):
    if not USE_INTERACTIVE_DESCENDANTS:
        return False, None, None
    node = nodes[node_index]
    if not dataset_visible(node) or is_interactive_candidate(node):
        return False, None, None

    ancestor_index, ancestor_depth = nearest_interactive_ancestor(
        node_index, nodes, parent_map
    )
    if ancestor_index is None:
        return False, None, None

    child_count = element_child_count(node)

    # Balanced preset: only direct leaf descendants are retained.  When a
    # nonzero descendant allowance is configured, known child counts
    # may be used, but unknown structure is not accepted automatically.
    if MAX_INTERACTIVE_DESCENDANT_CHILDREN <= 0:
        structurally_small = has_no_element_children(node)
    elif child_count is None:
        structurally_small = has_no_element_children(node)
    else:
        structurally_small = child_count <= MAX_INTERACTIVE_DESCENDANT_CHILDREN

    return structurally_small, ancestor_index, ancestor_depth


def extract_base_candidate_indices(
    new_nodes: List[Dict[str, Any]],
    parent_map: Dict[int, Optional[int]],
):
    """Page-wide generic pool before the query-specific fallback."""
    interactive = [
        index for index, node in enumerate(new_nodes)
        if is_interactive_candidate(node)
    ]
    text_leaf = [
        index for index, node in enumerate(new_nodes)
        if is_generic_text_leaf_candidate(node)
    ]

    interactive_descendants = []
    descendant_ancestor = {}
    descendant_depth = {}
    for index in range(len(new_nodes)):
        ok, ancestor_index, depth = is_interactive_descendant_candidate(
            index, new_nodes, parent_map
        )
        if ok:
            interactive_descendants.append(index)
            descendant_ancestor[index] = ancestor_index
            descendant_depth[index] = depth

    merged, kinds, seen = [], {}, set()
    ordered_sources = (
        [(index, "interactive") for index in interactive]
        + [(index, "text_leaf") for index in text_leaf]
        + [(index, "interactive_descendant") for index in interactive_descendants]
    )
    for index, kind in ordered_sources:
        if index in seen:
            continue
        seen.add(index)
        merged.append(index)
        kinds[index] = kind

    stats = {
        "interactive_count": len(interactive),
        "text_leaf_count": len(text_leaf),
        "interactive_descendant_count": len(interactive_descendants),
        "descendant_ancestor": descendant_ancestor,
        "descendant_depth": descendant_depth,
    }
    return merged, kinds, stats


def node_tag_family(tag: str) -> str:
    tag = normalise_text(tag)
    if tag in {"input", "textarea", "select", "button", "option"}:
        return "form_control"
    if tag == "a":
        return "link"
    if tag in {"img", "svg", "picture", "canvas", "video", "audio", "i"}:
        return "media_or_icon"
    if tag in {"h1", "h2", "h3", "h4", "h5", "h6", "p", "span", "label", "strong", "em", "small", "b", "bdi"}:
        return "text"
    if tag in {"li", "ul", "ol", "menu"}:
        return "list"
    if tag in {"td", "th", "tr", "table", "thead", "tbody"}:
        return "table"
    if tag in {"div", "section", "article", "main", "aside", "header", "footer", "nav", "form"}:
        return "container"
    if "-" in tag:
        return "custom_element"
    return "other"


def tag_compatibility(old_node: Dict[str, Any], new_node: Dict[str, Any]) -> float:
    old_tag, new_tag = node_tag(old_node), node_tag(new_node)
    if old_tag and old_tag == new_tag:
        return 1.0
    if old_tag and new_tag and node_tag_family(old_tag) == node_tag_family(new_tag):
        return 0.65
    return 0.0


def role_compatibility(old_node: Dict[str, Any], new_node: Dict[str, Any]) -> float:
    old_role = normalise_text(node_attribute(old_node, "role"))
    new_role = normalise_text(node_attribute(new_node, "role"))
    if old_role and old_role == new_role:
        return 1.0
    if old_role in INTERACTIVE_ROLES and new_role in INTERACTIVE_ROLES:
        return 0.5
    return 0.0


def attribute_similarity(old_node: Dict[str, Any], new_node: Dict[str, Any]) -> float:
    similarities = []
    for names in [
        ("id",),
        ("data-testid", "data-test", "data-qa"),
        ("name",),
        ("class",),
        ("aria-label",),
        ("title",),
        ("placeholder",),
    ]:
        old_value = node_attribute(old_node, *names)
        new_value = node_attribute(new_node, *names)
        if not old_value or not new_value:
            continue
        similarities.append(max(
            sequence_similarity(old_value, new_value),
            token_jaccard(old_value, new_value),
        ))
    return max(similarities, default=0.0)


def geometry_similarity(old_node: Dict[str, Any], new_node: Dict[str, Any]) -> float:
    scores = []
    for names in [(["width", "bboxW", "w"]), (["height", "bboxH", "h"])]:
        old_value = first_number(old_node, names, 0.0)
        new_value = first_number(new_node, names, 0.0)
        if old_value > 0 and new_value > 0:
            scores.append(min(old_value, new_value) / max(old_value, new_value))
    return float(np.mean(scores)) if scores else 0.0


def fallback_similarity(
    old_node: Dict[str, Any],
    new_node: Dict[str, Any],
    old_index: int,
    new_index: int,
) -> float:
    text_score = max(
        sequence_similarity(node_text(old_node), node_text(new_node)),
        token_jaccard(node_text(old_node), node_text(new_node)),
    )
    attr_score = attribute_similarity(old_node, new_node)
    tag_score = tag_compatibility(old_node, new_node)
    role_score = role_compatibility(old_node, new_node)
    xpath_score = token_jaccard(node_xpath(old_node), node_xpath(new_node))
    geometry_score = geometry_similarity(old_node, new_node)
    index_score = 1.0 / (1.0 + float(np.log1p(abs(new_index - old_index))))

    return float(
        0.35 * text_score
        + 0.20 * attr_score
        + 0.10 * tag_score
        + 0.10 * role_score
        + 0.10 * xpath_score
        + 0.10 * geometry_score
        + 0.05 * index_score
    )


def select_query_fallback_candidates(
    old_node: Dict[str, Any],
    old_index: int,
    new_nodes: List[Dict[str, Any]],
    excluded_indices: Iterable[int],
):
    """Rank rejected visible nodes without consulting the true_new_idx label."""
    if not USE_QUERY_SPECIFIC_FALLBACK or FALLBACK_TOP_K <= 0:
        return [], {}, {}, 0

    excluded = set(excluded_indices)
    ranked = []
    for new_index, new_node in enumerate(new_nodes):
        if new_index in excluded or not dataset_visible(new_node):
            continue
        score = fallback_similarity(old_node, new_node, old_index, new_index)
        ranked.append((score, new_index))

    ranked.sort(key=lambda item: (-item[0], item[1]))
    score_by_index = {index: float(score) for score, index in ranked}
    rank_by_index = {index: rank for rank, (_, index) in enumerate(ranked, start=1)}
    selected = [
        index for score, index in ranked
        if score >= FALLBACK_MIN_SCORE
    ][:FALLBACK_TOP_K]
    return selected, score_by_index, rank_by_index, len(ranked)


def merge_query_candidates(
    base_indices: List[int],
    base_kinds: Dict[int, str],
    fallback_indices: List[int],
    cap: Optional[int],
):
    merged, kinds, seen = [], {}, set()
    for index, kind in (
        [(index, base_kinds.get(index, "base")) for index in base_indices]
        + [(index, "fallback") for index in fallback_indices]
    ):
        if index in seen:
            continue
        seen.add(index)
        merged.append(index)
        kinds[index] = kind

    uncapped = list(merged)
    if cap is not None:
        merged = merged[:cap]
        kinds = {index: kinds[index] for index in merged}
    return merged, kinds, uncapped


def exact_shortlist_protection_reason(
    old_node: Dict[str, Any],
    new_node: Dict[str, Any],
) -> str:
    """Return label-independent reasons for preserving a candidate."""
    reasons = []

    if PROTECT_EXACT_ID:
        old_value = normalise_text(node_attribute(old_node, "id"))
        new_value = normalise_text(node_attribute(new_node, "id"))
        if old_value and old_value == new_value:
            reasons.append("exact_id")

    if PROTECT_EXACT_DATA_TEST_ID:
        old_value = normalise_text(
            node_attribute(old_node, "data-testid", "data-test", "data-qa")
        )
        new_value = normalise_text(
            node_attribute(new_node, "data-testid", "data-test", "data-qa")
        )
        if old_value and old_value == new_value:
            reasons.append("exact_data_test_id")

    if PROTECT_EXACT_NAME:
        old_value = normalise_text(node_attribute(old_node, "name"))
        new_value = normalise_text(node_attribute(new_node, "name"))
        if old_value and old_value == new_value:
            reasons.append("exact_name")

    if PROTECT_EXACT_ACCESSIBLE_TEXT:
        old_value = accessible_text(old_node)
        new_value = accessible_text(new_node)
        if (
            old_value
            and len(old_value) >= MIN_ACCESSIBLE_TEXT_LENGTH
            and old_value == new_value
        ):
            reasons.append("exact_accessible_text")

    return "|".join(reasons)


def rank_query_candidate_pool(
    old_node: Dict[str, Any],
    old_index: int,
    new_nodes: List[Dict[str, Any]],
    candidate_indices: List[int],
):
    """Rank the full retrieved pool without using the ground-truth label."""
    scored = []
    protection_by_index = {}

    for new_index in candidate_indices:
        new_node = new_nodes[new_index]
        score = fallback_similarity(
            old_node=old_node,
            new_node=new_node,
            old_index=old_index,
            new_index=new_index,
        )
        scored.append((float(score), int(new_index)))

        reason = exact_shortlist_protection_reason(old_node, new_node)
        if reason:
            protection_by_index[new_index] = reason

    scored.sort(key=lambda item: (-item[0], item[1]))
    ranked_indices = [index for _, index in scored]
    score_by_index = {index: score for score, index in scored}
    rank_by_index = {
        index: rank
        for rank, index in enumerate(ranked_indices, start=1)
    }
    return (
        ranked_indices,
        score_by_index,
        rank_by_index,
        protection_by_index,
    )


def apply_query_shortlist(
    full_candidate_indices: List[int],
    ranked_candidate_indices: List[int],
    protection_by_index: Dict[int, str],
    top_k: Optional[int],
) -> List[int]:
    """
    Keep the top-K cheap retrieval results plus exact protected matches.

    The output follows the original balanced-pool order so candidate order is
    stable and does not directly encode the cheap retrieval rank.
    """
    if top_k is None:
        return list(full_candidate_indices)

    top_k = max(0, int(top_k))
    selected = set(ranked_candidate_indices[:top_k])
    selected.update(protection_by_index.keys())

    return [
        index for index in full_candidate_indices
        if index in selected
    ]


def shortlist_label(top_k: Optional[int]) -> str:
    return "full" if top_k is None else str(int(top_k))


def diagnose_positive_candidate(
    true_new_index: int,
    old_node: Dict[str, Any],
    old_index: int,
    new_nodes: List[Dict[str, Any]],
    parent_map: Dict[int, Optional[int]],
    base_indices: List[int],
    base_kinds: Dict[int, str],
    fallback_indices: List[int],
    fallback_score_by_index: Dict[int, float],
    fallback_rank_by_index: Dict[int, int],
    uncapped_final_indices: List[int],
    final_indices: List[int],
    final_kinds: Dict[int, str],
) -> Dict[str, Any]:
    """Explain why the official positive was retained or rejected."""
    if true_new_index < 0 or true_new_index >= len(new_nodes):
        return {
            "reject_reason": "true_new_idx_out_of_range",
            "eligible_before_cap": False,
            "uncapped_candidate_rank": None,
            "positive_retained": False,
            "positive_candidate_source": "",
            "positive_tag": "",
            "positive_role": "",
            "positive_text": "",
            "positive_accessible_text": "",
            "positive_accessible_text_length": 0,
            "positive_displayed": False,
            "positive_width": np.nan,
            "positive_height": np.nan,
            "interactive_ok": False,
            "text_leaf_ok": False,
            "interactive_descendant_ok": False,
            "interactive_ancestor_index": None,
            "interactive_ancestor_depth": None,
            "leaf_ok": False,
            "base_candidate_ok": False,
            "fallback_selected": False,
            "fallback_rank": None,
            "fallback_score": np.nan,
            "positive_href": "",
            "positive_aria_label": "",
            "positive_onclick": "",
            "positive_tabindex": "",
            "positive_contenteditable": "",
        }

    node = new_nodes[true_new_index]
    displayed = first_bool(
        node,
        ["isDisplayed", "displayed", "isVisible", "visible"],
        default=True,
    )
    width = first_number(node, ["width", "bboxW", "w"], 0.0)
    height = first_number(node, ["height", "bboxH", "h"], 0.0)
    accessible = accessible_text(node)

    interactive_ok = is_interactive_candidate(node)
    text_leaf_ok = is_generic_text_leaf_candidate(node)
    descendant_ok, ancestor_index, ancestor_depth = is_interactive_descendant_candidate(
        true_new_index, new_nodes, parent_map
    )
    base_candidate_ok = true_new_index in base_indices
    fallback_selected = true_new_index in fallback_indices
    eligible_before_cap = true_new_index in uncapped_final_indices
    retained = true_new_index in final_indices

    uncapped_rank = (
        uncapped_final_indices.index(true_new_index) + 1
        if eligible_before_cap else None
    )
    fallback_rank = fallback_rank_by_index.get(true_new_index)
    fallback_score = fallback_score_by_index.get(true_new_index, np.nan)

    if retained:
        reason = "retained"
    elif not displayed:
        reason = "not_displayed"
    elif width < 2.0 or height < 2.0:
        reason = "bbox_too_small_or_missing"
    elif eligible_before_cap:
        reason = "exceeds_candidate_cap"
    elif fallback_rank is None:
        reason = "not_in_visible_fallback_pool"
    elif fallback_score < FALLBACK_MIN_SCORE:
        reason = "fallback_score_below_threshold"
    elif fallback_rank > FALLBACK_TOP_K:
        reason = "outside_fallback_top_k"
    else:
        reason = "not_selected_by_generalized_retrieval"

    return {
        "reject_reason": reason,
        "eligible_before_cap": eligible_before_cap,
        "uncapped_candidate_rank": uncapped_rank,
        "positive_retained": retained,
        "positive_candidate_source": final_kinds.get(true_new_index, ""),
        "positive_tag": node_tag(node),
        "positive_role": normalise_text(node_attribute(node, "role")),
        "positive_text": raw_node_text(node),
        "positive_accessible_text": accessible,
        "positive_accessible_text_length": len(accessible),
        "positive_displayed": displayed,
        "positive_width": width,
        "positive_height": height,
        "interactive_ok": interactive_ok,
        "text_leaf_ok": text_leaf_ok,
        "interactive_descendant_ok": bool(descendant_ok),
        "interactive_ancestor_index": ancestor_index,
        "interactive_ancestor_depth": ancestor_depth,
        "leaf_ok": has_no_element_children(node),
        "base_candidate_ok": base_candidate_ok,
        "fallback_selected": fallback_selected,
        "fallback_rank": fallback_rank,
        "fallback_score": fallback_score,
        "positive_href": node_attribute(node, "href"),
        "positive_aria_label": node_attribute(node, "aria-label"),
        "positive_onclick": node_attribute(node, "onclick"),
        "positive_tabindex": node_attribute(node, "tabindex"),
        "positive_contenteditable": node_attribute(node, "contenteditable"),
    }


# Runtime-aligned deterministic feature helpers
# These functions mirror the deterministic feature definitions in app.py.
# They use only information available in old/new domNodeInfo.json files.

RUNTIME_XPATH_TOKEN_RE = re.compile(r"[A-Za-z0-9_-]+")


def runtime_norm(value: Any) -> str:
    text = "" if value is None else str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split()).strip()


def runtime_token_set(value: Any) -> set:
    return {
        token for token in runtime_norm(value).split()
        if len(token) >= 2
    }


def runtime_seq_sim(left: Any, right: Any) -> float:
    left_text, right_text = runtime_norm(left), runtime_norm(right)
    if not left_text and not right_text:
        return 1.0
    if not left_text or not right_text:
        return 0.0
    return float(SequenceMatcher(None, left_text, right_text).ratio())


def runtime_jaccard(left: Any, right: Any) -> float:
    left_tokens = runtime_token_set(left)
    right_tokens = runtime_token_set(right)
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


def runtime_class_jaccard(left: Any, right: Any) -> float:
    left_tokens = {
        token for token in runtime_norm(left).split() if token
    }
    right_tokens = {
        token for token in runtime_norm(right).split() if token
    }
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


def runtime_xpath_token_sim(left: Any, right: Any) -> float:
    def tokens(value: Any) -> set:
        if not value:
            return set()
        return {
            runtime_norm(token)
            for token in RUNTIME_XPATH_TOKEN_RE.findall(str(value))
            if len(runtime_norm(token)) >= 2
        }

    left_tokens, right_tokens = tokens(left), tokens(right)
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


def runtime_node_attr(node: Dict[str, Any], key: str) -> str:
    """Prefer the actual top-level DOM field, then the attributes map."""
    aliases = {
        "id": ["id"],
        "name": ["name"],
        "class": ["className", "classname", "class"],
        "placeholder": ["placeholder"],
        "type": ["type"],
        "value": ["value"],
        "aria-label": ["ariaLabel", "arialabel", "aria-label"],
        "data-testid": ["dataTestId", "datatestid", "data-testid", "data-test", "data-qa"],
        "role": ["role"],
        "title": ["title"],
        "for": ["htmlFor", "for"],
        "label": ["labelText", "labeltext", "label"],
    }
    for alias in aliases.get(key, [key]):
        value = node.get(alias)
        if value is not None and str(value).strip() != "":
            return str(value)

    if key == "data-testid":
        return node_attribute(node, "data-testid", "data-test", "data-qa")
    if key == "aria-label":
        return node_attribute(node, "aria-label", "arialabel")
    if key == "class":
        return node_attribute(node, "class", "classname")
    return node_attribute(node, key)


def node_depth_value(node: Dict[str, Any]) -> float:
    for key in ["depth", "domDepth", "level"]:
        value = first_number(node, [key], np.nan)
        if np.isfinite(value):
            return float(value)
    dom_path = node.get("domPath")
    if isinstance(dom_path, list):
        return float(len(dom_path))
    return 0.0


def node_visible_value(node: Dict[str, Any]) -> bool:
    return first_bool(
        node,
        ["isDisplayed", "isVisible", "displayed", "visible"],
        default=True,
    )


def node_enabled_value(node: Dict[str, Any]) -> bool:
    return first_bool(
        node,
        ["isEnable", "isEnabled", "enabled"],
        default=True,
    )


def node_bbox(node: Dict[str, Any]):
    x = first_number(node, ["x", "bboxX", "left"], np.nan)
    y = first_number(node, ["y", "bboxY", "top"], np.nan)
    width = first_number(node, ["width", "bboxW", "w"], np.nan)
    height = first_number(node, ["height", "bboxH", "h"], np.nan)

    if not all(np.isfinite(value) for value in [x, y, width, height]):
        return None
    if width <= 0 or height <= 0:
        return None
    return float(x), float(y), float(width), float(height)


def runtime_geometry_features(
    old_node: Dict[str, Any],
    candidate_node: Dict[str, Any],
) -> Dict[str, Any]:
    old_box = node_bbox(old_node)
    candidate_box = node_bbox(candidate_node)

    if old_box is None or candidate_box is None:
        return {
            "geo_iou": 0.0,
            "geo_center_dist": 0.0,
            "geo_size_ratio": 0.0,
            "bbox_dx": 0.0,
            "bbox_dy": 0.0,
            "bbox_area_ratio": 0.0,
            "geometry_available": 0,
            "old_bbox_x": np.nan,
            "old_bbox_y": np.nan,
            "old_bbox_w": np.nan,
            "old_bbox_h": np.nan,
            "new_bbox_x": np.nan,
            "new_bbox_y": np.nan,
            "new_bbox_w": np.nan,
            "new_bbox_h": np.nan,
        }

    ox, oy, ow, oh = old_box
    nx, ny, nw, nh = candidate_box

    old_center_x = ox + ow / 2.0
    old_center_y = oy + oh / 2.0
    new_center_x = nx + nw / 2.0
    new_center_y = ny + nh / 2.0

    center_distance = float(np.sqrt(
        (old_center_x - new_center_x) ** 2
        + (old_center_y - new_center_y) ** 2
    ))

    intersection_width = max(
        0.0,
        min(ox + ow, nx + nw) - max(ox, nx),
    )
    intersection_height = max(
        0.0,
        min(oy + oh, ny + nh) - max(oy, ny),
    )
    intersection = intersection_width * intersection_height

    old_area = ow * oh
    new_area = nw * nh
    union = old_area + new_area - intersection + 1e-6
    area_ratio = old_area / (new_area + 1e-6)

    return {
        "geo_iou": float(np.clip(intersection / union, 0.0, 1.0)),
        "geo_center_dist": center_distance,
        "geo_size_ratio": float(np.log1p(area_ratio)),
        "bbox_dx": float(abs(ox - nx)),
        "bbox_dy": float(abs(oy - ny)),
        "bbox_area_ratio": float(np.clip(area_ratio, 0.0, 100.0)),
        "geometry_available": 1,
        "old_bbox_x": ox,
        "old_bbox_y": oy,
        "old_bbox_w": ow,
        "old_bbox_h": oh,
        "new_bbox_x": nx,
        "new_bbox_y": ny,
        "new_bbox_w": nw,
        "new_bbox_h": nh,
    }


def build_runtime_context_maps(
    nodes: List[Dict[str, Any]],
    parent_map: Dict[int, Optional[int]],
) -> Dict[str, Dict[int, Any]]:
    """
    Build deterministic text and structural context from the grouped UITestFix
    DOM. All indices refer to domNodeInfo.json entries.
    """
    children_map: Dict[Optional[int], List[int]] = defaultdict(list)
    for index, parent_index in parent_map.items():
        if parent_index is not None and 0 <= parent_index < len(nodes):
            children_map[parent_index].append(index)

    for children in children_map.values():
        children.sort()

    heading_tags = {"h1", "h2", "h3", "h4", "h5", "h6"}

    label_for_text: Dict[str, str] = {}
    for node in nodes:
        if node_tag(node) != "label":
            continue
        target = runtime_norm(runtime_node_attr(node, "for"))
        text = raw_node_text(node)
        if target and text:
            label_for_text.setdefault(target, text)

    def sibling_texts(index: int, direction: str, limit: int = 2) -> str:
        parent_index = parent_map.get(index)
        siblings = children_map.get(parent_index, [])
        if index not in siblings:
            return ""
        position = siblings.index(index)
        if direction == "before":
            selected = list(reversed(siblings[max(0, position - limit):position]))
        else:
            selected = siblings[position + 1:position + 1 + limit]
        texts = [raw_node_text(nodes[item]) for item in selected]
        return " | ".join(text for text in texts if text)

    def first_heading_descendant(root_index: int) -> str:
        queue = list(children_map.get(root_index, []))
        visited = set()
        while queue:
            current = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)
            if node_tag(nodes[current]) in heading_tags:
                text = raw_node_text(nodes[current])
                if text:
                    return text
            queue.extend(children_map.get(current, []))
        return ""

    contexts = {
        "parent_text": {},
        "parent_tag": {},
        "sibling_text": {},
        "sibling_before": {},
        "sibling_after": {},
        "child_text": {},
        "context_text": {},
        "nearby_text": {},
        "heading_context": {},
        "ancestor_chain": {},
        "label_text": {},
        "dom_depth": {},
        # Structural context used by the eight structural model features.
        "ancestor_tags_nearest": {},
        "ancestor_tags_root": {},
        "sibling_tags": {},
        "child_tags": {},
        "sibling_position": {},
        "sibling_count": {},
        "parent_map": parent_map,
        "children_map": dict(children_map),
    }

    for index, node in enumerate(nodes):
        parent_index = parent_map.get(index)
        parent_node = (
            nodes[parent_index]
            if parent_index is not None and 0 <= parent_index < len(nodes)
            else None
        )

        parent_text = raw_node_text(parent_node)[:200] if parent_node else ""
        parent_tag = node_tag(parent_node) if parent_node else ""

        sibling_before = sibling_texts(index, "before", limit=2)
        sibling_after = sibling_texts(index, "after", limit=2)
        sibling_text = " | ".join(
            value for value in [sibling_before, sibling_after] if value
        )

        direct_children = children_map.get(index, [])
        child_values = [
            raw_node_text(nodes[child])
            for child in direct_children[:3]
        ]
        child_text = " | ".join(value for value in child_values if value)

        nearby_text = parent_text[:120]

        context_parts = [
            raw_node_text(node),
            parent_text,
            sibling_text,
            child_text,
            runtime_node_attr(node, "aria-label"),
            runtime_node_attr(node, "placeholder"),
            runtime_node_attr(node, "title"),
            runtime_node_attr(node, "value"),
        ]
        context_text = normalise_text(
            " ".join(value for value in context_parts if value)
        )

        heading_context = ""
        current = parent_index
        depth = 0
        while current is not None and 0 <= current < len(nodes) and depth < 8:
            current_node = nodes[current]
            if node_tag(current_node) in heading_tags:
                heading_context = raw_node_text(current_node)
                if heading_context:
                    break
            heading_context = first_heading_descendant(current)
            if heading_context:
                break
            current = parent_map.get(current)
            depth += 1

        ancestor_parts = []
        ancestor_tags_nearest = []
        current = parent_index
        depth = 0
        while current is not None and 0 <= current < len(nodes) and depth < 12:
            current_node = nodes[current]
            tag = node_tag(current_node)
            class_tokens = runtime_norm(runtime_node_attr(current_node, "class")).split()
            first_class = class_tokens[0] if class_tokens else ""
            if tag:
                ancestor_tags_nearest.append(tag)
                if depth < 5:
                    ancestor_parts.append(
                        f"{tag}.{first_class}" if first_class else tag
                    )
            current = parent_map.get(current)
            depth += 1

        label_text = ""
        current = index
        depth = 0
        while current is not None and 0 <= current < len(nodes) and depth < 8:
            current_node = nodes[current]
            if node_tag(current_node) == "label":
                label_text = raw_node_text(current_node)
                if label_text:
                    break
            current = parent_map.get(current)
            depth += 1

        if not label_text:
            node_id = runtime_norm(runtime_node_attr(node, "id"))
            label_text = label_for_text.get(node_id, "")

        if not label_text:
            label_text = runtime_node_attr(node, "label")

        siblings = children_map.get(parent_index, [])
        sibling_position = 0.0
        if index in siblings and len(siblings) > 1:
            sibling_position = siblings.index(index) / (len(siblings) - 1)

        sibling_indices = []
        if index in siblings:
            position = siblings.index(index)
            sibling_indices = (
                siblings[max(0, position - 2):position]
                + siblings[position + 1:position + 3]
            )

        contexts["parent_text"][index] = parent_text
        contexts["parent_tag"][index] = parent_tag
        contexts["sibling_text"][index] = sibling_text
        contexts["sibling_before"][index] = sibling_before
        contexts["sibling_after"][index] = sibling_after
        contexts["child_text"][index] = child_text
        contexts["context_text"][index] = context_text
        contexts["nearby_text"][index] = nearby_text
        contexts["heading_context"][index] = heading_context
        contexts["ancestor_chain"][index] = " > ".join(ancestor_parts)
        contexts["label_text"][index] = label_text
        contexts["dom_depth"][index] = node_depth_value(node)
        contexts["ancestor_tags_nearest"][index] = ancestor_tags_nearest
        contexts["ancestor_tags_root"][index] = list(reversed(ancestor_tags_nearest))
        contexts["sibling_tags"][index] = [
            node_tag(nodes[sibling])
            for sibling in sibling_indices
            if node_tag(nodes[sibling])
        ]
        contexts["child_tags"][index] = [
            node_tag(nodes[child])
            for child in direct_children
            if node_tag(nodes[child])
        ]
        contexts["sibling_position"][index] = float(sibling_position)
        contexts["sibling_count"][index] = int(len(siblings))

    return contexts


def xpath_tag_sequence(xpath: Any) -> List[str]:
    """Return an index-free, root-to-leaf tag sequence from an XPath."""
    xpath = normalise_text(xpath)
    if not xpath:
        return []
    sequence = []
    for segment in xpath.split("/"):
        segment = re.sub(r"\[.*?\]", "", segment).strip()
        segment = segment.split("::")[-1].strip()
        match = re.match(r"[a-zA-Z][a-zA-Z0-9_-]*", segment)
        if match:
            sequence.append(match.group(0).lower())
    return sequence


def ordered_sequence_similarity(left: Iterable[Any], right: Iterable[Any]) -> float:
    left_values = list(left or [])
    right_values = list(right or [])
    if not left_values and not right_values:
        return 1.0
    if not left_values or not right_values:
        return 0.0
    return float(SequenceMatcher(None, left_values, right_values).ratio())


def lcs_similarity(left: Iterable[Any], right: Iterable[Any]) -> float:
    """Order-aware similarity normalised by the longer sequence."""
    a, b = list(left or []), list(right or [])
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    previous = [0] * (len(b) + 1)
    for left_value in a:
        current = [0]
        for column, right_value in enumerate(b, start=1):
            if left_value == right_value:
                current.append(previous[column - 1] + 1)
            else:
                current.append(max(previous[column], current[-1]))
        previous = current
    return float(previous[-1] / max(len(a), len(b)))


def sequence_jaccard(left: Iterable[Any], right: Iterable[Any]) -> float:
    left_set = {str(value) for value in (left or []) if str(value)}
    right_set = {str(value) for value in (right or []) if str(value)}
    if not left_set and not right_set:
        return 1.0
    if not left_set or not right_set:
        return 0.0
    return float(len(left_set & right_set) / len(left_set | right_set))


def runtime_structural_features(
    old_node: Dict[str, Any],
    candidate_node: Dict[str, Any],
    old_index: int,
    candidate_index: int,
    old_context: Dict[str, Dict[int, Any]],
    new_context: Dict[str, Dict[int, Any]],
) -> Dict[str, float]:
    """
    Eight genuinely structural features. Attribute values such as id, class,
    and aria-label are intentionally excluded from this group.
    """
    old_depth = float(old_context["dom_depth"].get(old_index, 0.0))
    new_depth = float(new_context["dom_depth"].get(candidate_index, 0.0))
    depth_denominator = max(old_depth, new_depth, 1.0)

    old_ancestor_nearest = old_context["ancestor_tags_nearest"].get(
        old_index, []
    )
    new_ancestor_nearest = new_context["ancestor_tags_nearest"].get(
        candidate_index, []
    )
    old_ancestor_root = old_context["ancestor_tags_root"].get(old_index, [])
    new_ancestor_root = new_context["ancestor_tags_root"].get(
        candidate_index, []
    )

    old_xpath_tags = xpath_tag_sequence(node_xpath(old_node))
    new_xpath_tags = xpath_tag_sequence(node_xpath(candidate_node))

    old_sibling_position = float(
        old_context["sibling_position"].get(old_index, 0.0)
    )
    new_sibling_position = float(
        new_context["sibling_position"].get(candidate_index, 0.0)
    )

    return {
        "tag_match": float(
            bool(node_tag(old_node))
            and node_tag(old_node) == node_tag(candidate_node)
        ),
        "dom_depth_norm_diff": float(
            np.clip(abs(old_depth - new_depth) / depth_denominator, 0.0, 1.0)
        ),
        "parent_chain_similarity": ordered_sequence_similarity(
            old_ancestor_nearest[:4],
            new_ancestor_nearest[:4],
        ),
        "ancestor_path_similarity": lcs_similarity(
            old_ancestor_root,
            new_ancestor_root,
        ),
        "xpath_token_sim": sequence_jaccard(
            old_xpath_tags,
            new_xpath_tags,
        ),
        "sibling_position_diff": float(
            np.clip(
                abs(old_sibling_position - new_sibling_position),
                0.0,
                1.0,
            )
        ),
        "sibling_context_jaccard": sequence_jaccard(
            old_context["sibling_tags"].get(old_index, []),
            new_context["sibling_tags"].get(candidate_index, []),
        ),
        "child_tag_jaccard": sequence_jaccard(
            old_context["child_tags"].get(old_index, []),
            new_context["child_tags"].get(candidate_index, []),
        ),
    }


def runtime_attribute_features(
    old_node: Dict[str, Any],
    candidate_node: Dict[str, Any],
    old_index: int,
    candidate_index: int,
    old_context: Dict[str, Dict[int, Any]],
    new_context: Dict[str, Dict[int, Any]],
) -> Dict[str, Any]:
    old_id = runtime_node_attr(old_node, "id")
    new_id = runtime_node_attr(candidate_node, "id")
    old_data_test_id = runtime_node_attr(old_node, "data-testid")
    new_data_test_id = runtime_node_attr(candidate_node, "data-testid")
    old_name = runtime_node_attr(old_node, "name")
    new_name = runtime_node_attr(candidate_node, "name")
    old_class = runtime_node_attr(old_node, "class")
    new_class = runtime_node_attr(candidate_node, "class")

    id_subset_value = token_subset(old_id, new_id)
    data_subset_value = token_subset(old_data_test_id, new_data_test_id)
    name_subset_value = token_subset(old_name, new_name)

    old_visible = node_visible_value(old_node)
    new_visible = node_visible_value(candidate_node)
    old_enabled = node_enabled_value(old_node)
    new_enabled = node_enabled_value(candidate_node)

    old_depth = float(old_context["dom_depth"].get(old_index, 0.0))
    new_depth = float(new_context["dom_depth"].get(candidate_index, 0.0))

    return {
        "label_seqsim": runtime_seq_sim(
            old_context["label_text"].get(old_index, ""),
            new_context["label_text"].get(candidate_index, ""),
        ),
        "heading_seqsim": runtime_seq_sim(
            old_context["heading_context"].get(old_index, ""),
            new_context["heading_context"].get(candidate_index, ""),
        ),
        "aria_seqsim": runtime_seq_sim(
            runtime_node_attr(old_node, "aria-label"),
            runtime_node_attr(candidate_node, "aria-label"),
        ),
        "placeholder_seqsim": runtime_seq_sim(
            runtime_node_attr(old_node, "placeholder"),
            runtime_node_attr(candidate_node, "placeholder"),
        ),
        "title_seqsim": runtime_seq_sim(
            runtime_node_attr(old_node, "title"),
            runtime_node_attr(candidate_node, "title"),
        ),
        "id_seqsim": runtime_seq_sim(old_id, new_id),
        "name_seqsim": runtime_seq_sim(old_name, new_name),
        "class_seqsim": runtime_seq_sim(old_class, new_class),
        "ancestor_jaccard": runtime_jaccard(
            old_context["ancestor_chain"].get(old_index, ""),
            new_context["ancestor_chain"].get(candidate_index, ""),
        ),
        "parent_seqsim": runtime_seq_sim(
            old_context["parent_text"].get(old_index, ""),
            new_context["parent_text"].get(candidate_index, ""),
        ),
        "sibling_before_jaccard": runtime_jaccard(
            old_context["sibling_text"].get(old_index, ""),
            new_context["sibling_before"].get(candidate_index, ""),
        ),
        "sibling_after_jaccard": runtime_jaccard(
            old_context["sibling_text"].get(old_index, ""),
            new_context["sibling_after"].get(candidate_index, ""),
        ),
        "nearby_jaccard": runtime_jaccard(
            old_context["nearby_text"].get(old_index, ""),
            new_context["nearby_text"].get(candidate_index, ""),
        ),
        "role_match": int(
            bool(runtime_norm(runtime_node_attr(old_node, "role")))
            and runtime_norm(runtime_node_attr(old_node, "role"))
            == runtime_norm(runtime_node_attr(candidate_node, "role"))
        ),
        "type_match": int(
            bool(runtime_norm(runtime_node_attr(old_node, "type")))
            and runtime_norm(runtime_node_attr(old_node, "type"))
            == runtime_norm(runtime_node_attr(candidate_node, "type"))
        ),
        "visible_match": int(old_visible == new_visible),
        "enabled_match": int(old_enabled == new_enabled),
        "id_subset": id_subset_value,
        "dataTestId_subset": data_subset_value,
        "attr_subset_any": int(
            bool(id_subset_value)
            or bool(data_subset_value)
            or bool(name_subset_value)
        ),
        "old_visible": int(old_visible),
        "new_visible": int(new_visible),
        "old_enabled": int(old_enabled),
        "new_enabled": int(new_enabled),
        "old_dom_depth": old_depth,
        "new_dom_depth": new_depth,
    }


ATTRIBUTE_FEATURE_COLUMNS = [
    "label_seqsim", "heading_seqsim", "aria_seqsim",
    "placeholder_seqsim", "title_seqsim", "id_seqsim", "name_seqsim",
    "class_seqsim",
    "ancestor_jaccard", "parent_seqsim",
    "sibling_before_jaccard", "sibling_after_jaccard", "nearby_jaccard",
    "role_match", "type_match", "visible_match", "enabled_match",
    "id_subset", "dataTestId_subset", "attr_subset_any",
]

GEOMETRY_FEATURE_COLUMNS = [
    "geo_iou", "geo_center_dist", "geo_size_ratio",
    "bbox_dx", "bbox_dy",
]

MODEL_LEXICAL_FEATURES = [
    "idx_distance_log",
    "old_text_len", "new_text_len", "len_diff", "text_len_ratio",
    "text_exact", "text_substr", "old_text_empty", "new_text_empty",
    "text_token_overlap", "text_seqsim", "n_candidates",
]

MODEL_SEMANTIC_FEATURES = [
    "semantic_sim",
    "parent_text_sim", "sibling_text_sim", "child_text_sim", "context_text_sim",
]

MODEL_STRUCTURAL_FEATURES = [
    "tag_match",
    "dom_depth_norm_diff",
    "parent_chain_similarity",
    "ancestor_path_similarity",
    "xpath_token_sim",
    "sibling_position_diff",
    "sibling_context_jaccard",
    "child_tag_jaccard",
]

MODEL_ATTRIBUTE_FEATURES = ATTRIBUTE_FEATURE_COLUMNS
MODEL_GEOMETRY_FEATURES = GEOMETRY_FEATURE_COLUMNS

ALL_MODEL_FEATURES = (
    MODEL_LEXICAL_FEATURES
    + MODEL_SEMANTIC_FEATURES
    + MODEL_STRUCTURAL_FEATURES
    + MODEL_ATTRIBUTE_FEATURES
    + MODEL_GEOMETRY_FEATURES
)

assert len(ALL_MODEL_FEATURES) == 50
assert len(ALL_MODEL_FEATURES) == len(set(ALL_MODEL_FEATURES))

# All 50 model features are exported by this notebook.
MODEL_FEATURE_COLUMNS = list(ALL_MODEL_FEATURES)

PAIR_COLUMNS = [
    "qid", "site", "split", "old_idx", "cand_new_idx",
    "candidate_extract_idx", "candidate_kind",
    "source_interactive", "source_text_leaf",
    "source_interactive_descendant", "source_fallback",
    "fallback_score", "fallback_rank",
    "shortlist_score", "shortlist_rank",
    "shortlist_protected", "shortlist_protection_reason",
    "label", "relevance", "idx_distance", "old_text_n", "new_text_n",
    "old_parent_text_n", "new_parent_text_n",
    "old_sibling_text_n", "new_sibling_text_n",
    "old_child_text_n", "new_child_text_n",
    "old_context_text_n", "new_context_text_n",
    "old_text_len", "new_text_len", "len_diff",
    "text_len_ratio", "text_exact", "text_substr", "old_text_empty",
    "new_text_empty", "text_token_overlap", "text_seqsim",
    "idx_distance_log", "n_candidates",
    *MODEL_SEMANTIC_FEATURES,
    *MODEL_STRUCTURAL_FEATURES,
    *ATTRIBUTE_FEATURE_COLUMNS,
    *GEOMETRY_FEATURE_COLUMNS,
    # Retained only as diagnostics; not part of the 50-feature model.
    "bbox_area_ratio",
    "geometry_available",
    "old_bbox_x", "old_bbox_y", "old_bbox_w", "old_bbox_h",
    "new_bbox_x", "new_bbox_y", "new_bbox_w", "new_bbox_h",
    "old_visible", "new_visible", "old_enabled", "new_enabled",
    "old_dom_depth", "new_dom_depth",
]


def token_subset(old_value: Any, new_value: Any) -> int:
    old_tokens, new_tokens = token_set(old_value), token_set(new_value)
    return int(bool(old_tokens) and old_tokens.issubset(new_tokens))


def build_pair_rows(
    query,
    old_node,
    new_nodes,
    old_context,
    new_context,
    candidate_indices,
    candidate_kinds,
    fallback_score_by_index,
    fallback_rank_by_index,
    shortlist_score_by_index,
    shortlist_rank_by_index,
    shortlist_protection_by_index,
):
    true_new_index = int(query["true_new_idx"])
    candidates = [(index, new_nodes[index]) for index in candidate_indices]
    old_index = int(query["old_idx"])
    old_text = node_text(old_node)
    candidate_texts = [node_text(candidate) for _, candidate in candidates]
    text_scores = [
        sequence_similarity(old_text, text)
        for text in candidate_texts
    ]

    rows = []
    n_candidates = len(candidates)

    for extract_index, (
        (candidate_index, candidate),
        candidate_text,
        text_score,
    ) in enumerate(zip(
        candidates,
        candidate_texts,
        text_scores,
    )):
        label = int(candidate_index == true_new_index)
        index_distance = abs(candidate_index - old_index)
        old_length, new_length = len(old_text), len(candidate_text)
        maximum_length = max(old_length, new_length) or 1
        candidate_kind = candidate_kinds.get(candidate_index, "unknown")

        text_exact = int(bool(old_text) and old_text == candidate_text)
        text_substr = int(
            bool(old_text)
            and bool(candidate_text)
            and (
                old_text in candidate_text
                or candidate_text in old_text
            )
        )
        text_token_overlap = int(
            bool(
                runtime_token_set(old_text)
                & runtime_token_set(candidate_text)
            )
        )

        structural_features = runtime_structural_features(
            old_node=old_node,
            candidate_node=candidate,
            old_index=old_index,
            candidate_index=candidate_index,
            old_context=old_context,
            new_context=new_context,
        )
        attr_features = runtime_attribute_features(
            old_node=old_node,
            candidate_node=candidate,
            old_index=old_index,
            candidate_index=candidate_index,
            old_context=old_context,
            new_context=new_context,
        )
        geometry_features = runtime_geometry_features(
            old_node,
            candidate,
        )

        shortlist_rank = int(
            shortlist_rank_by_index.get(candidate_index, 0)
        )

        rows.append({
            "qid": query["qid"],
            "site": query["site"],
            "split": query["split"],
            "old_idx": old_index,
            "cand_new_idx": candidate_index,
            "candidate_extract_idx": extract_index,
            "candidate_kind": candidate_kind,
            "source_interactive": int(candidate_kind == "interactive"),
            "source_text_leaf": int(candidate_kind == "text_leaf"),
            "source_interactive_descendant": int(
                candidate_kind == "interactive_descendant"
            ),
            "source_fallback": int(candidate_kind == "fallback"),
            "fallback_score": float(
                fallback_score_by_index.get(candidate_index, 0.0)
            ),
            "fallback_rank": int(
                fallback_rank_by_index.get(candidate_index, 0)
            ),
            "shortlist_score": float(
                shortlist_score_by_index.get(candidate_index, 0.0)
            ),
            "shortlist_rank": shortlist_rank,
            "shortlist_protected": int(
                candidate_index in shortlist_protection_by_index
            ),
            "shortlist_protection_reason": (
                shortlist_protection_by_index.get(candidate_index, "")
            ),
            "label": label,
            "relevance": 3 if label else 0,
            "idx_distance": index_distance,
            "old_text_n": old_text,
            "new_text_n": candidate_text,
            "old_parent_text_n": old_context["parent_text"].get(old_index, ""),
            "new_parent_text_n": new_context["parent_text"].get(candidate_index, ""),
            "old_sibling_text_n": old_context["sibling_text"].get(old_index, ""),
            "new_sibling_text_n": new_context["sibling_text"].get(candidate_index, ""),
            "old_child_text_n": old_context["child_text"].get(old_index, ""),
            "new_child_text_n": new_context["child_text"].get(candidate_index, ""),
            "old_context_text_n": old_context["context_text"].get(old_index, ""),
            "new_context_text_n": new_context["context_text"].get(candidate_index, ""),
            "old_text_len": old_length,
            "new_text_len": new_length,
            "len_diff": abs(old_length - new_length),
            "text_len_ratio": (
                min(old_length, new_length) / maximum_length
            ),
            "text_exact": text_exact,
            "text_substr": text_substr,
            "old_text_empty": int(not old_text),
            "new_text_empty": int(not candidate_text),
            # Match runtime: binary shared-token signal.
            "text_token_overlap": text_token_overlap,
            "text_seqsim": text_score,
            "idx_distance_log": float(np.log1p(index_distance)),
            "n_candidates": n_candidates,
            **structural_features,
            **attr_features,
            **geometry_features,
        })

    return rows


## 5. Build candidate shortlists

The notebook first prepares the complete balanced candidate pool and cheap
query-specific ranking for every exact framework query. It then applies the
single value configured in `QUERY_SHORTLIST_TOP_K` to development and test
queries alike.


In [6]:
# Pass 1: prepare the complete balanced retrieval plan for every query.
query_plans = []
page_cache = {}
site_static_rows = []
pair_issue_rows = list(issue_rows)

for site in tqdm(sorted(ALL_SITES, key=str.lower), desc="Prepare sites"):
    site_queries = queries_df[queries_df["site"] == site].to_dict("records")
    site_dir = find_site_dir(DATASET_ROOT, site)
    if site_dir is None:
        continue

    try:
        old_nodes = read_dom_list(
            require_dataset_file(site_dir, "old/domNodeInfo.json")
        )
        new_nodes = read_dom_list(
            require_dataset_file(site_dir, "new/domNodeInfo.json")
        )
    except Exception as error:
        pair_issue_rows.append({
            "site": site,
            "split": split_for_site(site),
            "old_idx": "",
            "true_new_idx": "",
            "issue": f"pair_generation_load_error: {error}",
        })
        continue

    old_parent_map = build_parent_index_map(old_nodes)
    parent_map = build_parent_index_map(new_nodes)
    old_runtime_context = build_runtime_context_maps(
        old_nodes, old_parent_map
    )
    new_runtime_context = build_runtime_context_maps(
        new_nodes, parent_map
    )
    resolved_parent_links = sum(
        parent is not None for parent in parent_map.values()
    )
    base_indices, base_kinds, base_stats = extract_base_candidate_indices(
        new_nodes, parent_map
    )

    page_cache[site] = {
        "old_nodes": old_nodes,
        "new_nodes": new_nodes,
        "parent_map": parent_map,
        "old_parent_map": old_parent_map,
        "old_runtime_context": old_runtime_context,
        "new_runtime_context": new_runtime_context,
        "base_indices": base_indices,
        "base_kinds": base_kinds,
        "base_stats": base_stats,
        "resolved_parent_links": resolved_parent_links,
    }

    site_static_rows.append({
        "site": site,
        "split": split_for_site(site),
        "new_dom_nodes": len(new_nodes),
        "old_geometry_nodes": sum(node_bbox(node) is not None for node in old_nodes),
        "new_geometry_nodes": sum(node_bbox(node) is not None for node in new_nodes),
        "old_visible_nodes": sum(node_visible_value(node) for node in old_nodes),
        "new_visible_nodes": sum(node_visible_value(node) for node in new_nodes),
        "old_enabled_nodes": sum(node_enabled_value(node) for node in old_nodes),
        "new_enabled_nodes": sum(node_enabled_value(node) for node in new_nodes),
        "resolved_parent_links": resolved_parent_links,
        "interactive_matches": base_stats["interactive_count"],
        "text_leaf_matches": base_stats["text_leaf_count"],
        "interactive_descendant_matches": (
            base_stats["interactive_descendant_count"]
        ),
        "base_candidate_count": len(base_indices),
        "base_dom_fraction": (
            len(base_indices) / len(new_nodes) if new_nodes else 0.0
        ),
        "expected_framework_queries": len(site_queries),
    })

    for query in tqdm(site_queries, desc=site, leave=False):
        old_index = int(query["old_idx"])
        true_new_index = int(query["true_new_idx"])

        if old_index < 0 or old_index >= len(old_nodes):
            pair_issue_rows.append({
                "site": site,
                "split": query["split"],
                "old_idx": old_index,
                "true_new_idx": true_new_index,
                "issue": "old_idx_out_of_range_during_plan_generation",
            })
            continue

        old_node = old_nodes[old_index]

        (
            fallback_indices,
            fallback_score_by_index,
            fallback_rank_by_index,
            fallback_pool_size,
        ) = select_query_fallback_candidates(
            old_node=old_node,
            old_index=old_index,
            new_nodes=new_nodes,
            excluded_indices=base_indices,
        )

        (
            full_candidate_indices,
            full_candidate_kinds,
            uncapped_full_indices,
        ) = merge_query_candidates(
            base_indices=base_indices,
            base_kinds=base_kinds,
            fallback_indices=fallback_indices,
            cap=MAX_CANDIDATES,
        )

        (
            ranked_candidate_indices,
            shortlist_score_by_index,
            shortlist_rank_by_index,
            shortlist_protection_by_index,
        ) = rank_query_candidate_pool(
            old_node=old_node,
            old_index=old_index,
            new_nodes=new_nodes,
            candidate_indices=full_candidate_indices,
        )

        query_plans.append({
            "query": query,
            "site": site,
            "split": query["split"],
            "old_index": old_index,
            "true_new_index": true_new_index,
            "fallback_indices": fallback_indices,
            "fallback_score_by_index": fallback_score_by_index,
            "fallback_rank_by_index": fallback_rank_by_index,
            "fallback_pool_size": fallback_pool_size,
            "full_candidate_indices": full_candidate_indices,
            "full_candidate_kinds": full_candidate_kinds,
            "uncapped_full_indices": uncapped_full_indices,
            "ranked_candidate_indices": ranked_candidate_indices,
            "shortlist_score_by_index": shortlist_score_by_index,
            "shortlist_rank_by_index": shortlist_rank_by_index,
            "shortlist_protection_by_index": shortlist_protection_by_index,
        })

print("Prepared exact query plans:", len(query_plans))


# Manual shortlist configuration.
# No sweep and no automatic model-selection logic are performed.
selected_top_k = QUERY_SHORTLIST_TOP_K

if selected_top_k is not None:
    selected_top_k = int(selected_top_k)
    if selected_top_k <= 0:
        raise ValueError(
            "QUERY_SHORTLIST_TOP_K must be a positive integer or None."
        )

print("Manual shortlist top-K:", shortlist_label(selected_top_k))


# Apply the manual K and write the final pair CSV.

query_report_rows = []
generation_accumulator = defaultdict(lambda: {
    "positive_retained_queries": 0,
    "generated_queries": 0,
    "candidate_rows": 0,
    "full_candidate_counts": [],
    "shortlist_candidate_counts": [],
    "fallback_selected_counts": [],
    "fallback_pool_sizes": [],
    "protected_counts": [],
})

with WORKING_PAIR_CSV.open("w", encoding="utf-8", newline="") as output_handle:
    writer = csv.DictWriter(
        output_handle,
        fieldnames=PAIR_COLUMNS,
        extrasaction="ignore",
        quoting=csv.QUOTE_MINIMAL,
    )
    writer.writeheader()

    for plan in tqdm(query_plans, desc="Write shortlisted pairs"):
        query = plan["query"]
        site = plan["site"]
        page = page_cache[site]
        old_nodes = page["old_nodes"]
        new_nodes = page["new_nodes"]
        parent_map = page["parent_map"]
        old_runtime_context = page["old_runtime_context"]
        new_runtime_context = page["new_runtime_context"]
        base_indices = page["base_indices"]
        base_kinds = page["base_kinds"]

        old_index = plan["old_index"]
        true_new_index = plan["true_new_index"]
        old_node = old_nodes[old_index]

        candidate_indices = apply_query_shortlist(
            full_candidate_indices=plan["full_candidate_indices"],
            ranked_candidate_indices=plan["ranked_candidate_indices"],
            protection_by_index=plan[
                "shortlist_protection_by_index"
            ],
            top_k=selected_top_k,
        )
        candidate_kinds = {
            index: plan["full_candidate_kinds"][index]
            for index in candidate_indices
        }

        diagnosis = diagnose_positive_candidate(
            true_new_index=true_new_index,
            old_node=old_node,
            old_index=old_index,
            new_nodes=new_nodes,
            parent_map=parent_map,
            base_indices=base_indices,
            base_kinds=base_kinds,
            fallback_indices=plan["fallback_indices"],
            fallback_score_by_index=plan[
                "fallback_score_by_index"
            ],
            fallback_rank_by_index=plan[
                "fallback_rank_by_index"
            ],
            uncapped_final_indices=plan[
                "uncapped_full_indices"
            ],
            final_indices=candidate_indices,
            final_kinds=plan["full_candidate_kinds"],
        )

        positive_in_full_pool = (
            true_new_index in plan["full_candidate_indices"]
        )
        positive_retained = (
            true_new_index in candidate_indices
        )

        if positive_in_full_pool and not positive_retained:
            diagnosis["reject_reason"] = "outside_query_shortlist"

        diagnosis["positive_retained"] = positive_retained
        diagnosis["positive_in_full_balanced_pool"] = (
            positive_in_full_pool
        )
        diagnosis["positive_shortlist_score"] = (
            plan["shortlist_score_by_index"].get(
                true_new_index, np.nan
            )
        )
        diagnosis["positive_shortlist_rank"] = (
            plan["shortlist_rank_by_index"].get(
                true_new_index
            )
        )
        diagnosis["positive_shortlist_protected"] = (
            true_new_index
            in plan["shortlist_protection_by_index"]
        )
        diagnosis["positive_shortlist_protection_reason"] = (
            plan["shortlist_protection_by_index"].get(
                true_new_index, ""
            )
        )

        site_acc = generation_accumulator[site]
        site_acc["full_candidate_counts"].append(
            len(plan["full_candidate_indices"])
        )
        site_acc["shortlist_candidate_counts"].append(
            len(candidate_indices)
        )
        site_acc["fallback_selected_counts"].append(
            len(plan["fallback_indices"])
        )
        site_acc["fallback_pool_sizes"].append(
            plan["fallback_pool_size"]
        )
        site_acc["protected_counts"].append(
            len(plan["shortlist_protection_by_index"])
        )

        if positive_retained:
            site_acc["positive_retained_queries"] += 1

        query_report_rows.append({
            "qid": query["qid"],
            "site": site,
            "split": query["split"],
            "old_idx": old_index,
            "true_new_idx": true_new_index,
            "new_dom_nodes": len(new_nodes),
            "resolved_parent_links": page[
                "resolved_parent_links"
            ],
            "base_candidate_count": len(base_indices),
            "fallback_pool_size": plan["fallback_pool_size"],
            "fallback_selected_count": len(
                plan["fallback_indices"]
            ),
            "full_balanced_candidate_count": len(
                plan["full_candidate_indices"]
            ),
            "shortlist_top_k": shortlist_label(
                selected_top_k
            ),
            "shortlist_protected_count": len(
                plan["shortlist_protection_by_index"]
            ),
            "candidate_count": len(candidate_indices),
            "candidate_dom_fraction": (
                len(candidate_indices) / len(new_nodes)
                if new_nodes else 0.0
            ),
            "written_to_pair_csv": (
                positive_retained or FORCE_INCLUDE_POSITIVE
            ),
            **diagnosis,
        })

        effective_indices = list(candidate_indices)
        effective_kinds = dict(candidate_kinds)

        if (
            not positive_retained
            and FORCE_INCLUDE_POSITIVE
            and 0 <= true_new_index < len(new_nodes)
        ):
            effective_indices.append(true_new_index)
            effective_kinds[true_new_index] = "forced_positive"

        if not positive_retained and not FORCE_INCLUDE_POSITIVE:
            pair_issue_rows.append({
                "site": site,
                "split": query["split"],
                "old_idx": old_index,
                "true_new_idx": true_new_index,
                "issue": "positive_not_retained_by_query_shortlist",
                "reject_reason": diagnosis["reject_reason"],
                "positive_shortlist_rank": diagnosis[
                    "positive_shortlist_rank"
                ],
                "positive_shortlist_score": diagnosis[
                    "positive_shortlist_score"
                ],
            })
            continue

        rows = build_pair_rows(
            query=query,
            old_node=old_node,
            new_nodes=new_nodes,
            old_context=old_runtime_context,
            new_context=new_runtime_context,
            candidate_indices=effective_indices,
            candidate_kinds=effective_kinds,
            fallback_score_by_index=plan[
                "fallback_score_by_index"
            ],
            fallback_rank_by_index=plan[
                "fallback_rank_by_index"
            ],
            shortlist_score_by_index=plan[
                "shortlist_score_by_index"
            ],
            shortlist_rank_by_index=plan[
                "shortlist_rank_by_index"
            ],
            shortlist_protection_by_index=plan[
                "shortlist_protection_by_index"
            ],
        )

        positive_count = sum(int(row["label"]) for row in rows)
        if positive_count != 1:
            pair_issue_rows.append({
                "site": site,
                "split": query["split"],
                "old_idx": old_index,
                "true_new_idx": true_new_index,
                "issue": f"generated_positive_count_{positive_count}",
                "reject_reason": diagnosis["reject_reason"],
            })
            continue

        writer.writerows(rows)
        site_acc["generated_queries"] += 1
        site_acc["candidate_rows"] += len(rows)

query_report_df = pd.DataFrame(query_report_rows)
issues_df = pd.DataFrame(pair_issue_rows)
missing_positive_df = query_report_df[
    ~query_report_df["positive_retained"]
].copy()

generation_rows = []
static_by_site = {
    row["site"]: row for row in site_static_rows
}

for site in sorted(ALL_SITES, key=str.lower):
    static = static_by_site.get(site)
    if static is None:
        continue

    acc = generation_accumulator[site]
    query_count = int(static["expected_framework_queries"])
    shortlisted_counts = acc["shortlist_candidate_counts"]
    full_counts = acc["full_candidate_counts"]

    generation_rows.append({
        **static,
        "query_shortlist_top_k": shortlist_label(
            selected_top_k
        ),
        "mean_full_balanced_candidates": (
            float(np.mean(full_counts)) if full_counts else 0.0
        ),
        "mean_shortlist_candidates": (
            float(np.mean(shortlisted_counts))
            if shortlisted_counts else 0.0
        ),
        "median_shortlist_candidates": (
            float(np.median(shortlisted_counts))
            if shortlisted_counts else 0.0
        ),
        "max_shortlist_candidates": (
            int(max(shortlisted_counts))
            if shortlisted_counts else 0
        ),
        "mean_shortlist_dom_fraction": (
            float(np.mean(shortlisted_counts))
            / int(static["new_dom_nodes"])
            if shortlisted_counts and static["new_dom_nodes"]
            else 0.0
        ),
        "mean_fallback_selected": (
            float(np.mean(acc["fallback_selected_counts"]))
            if acc["fallback_selected_counts"] else 0.0
        ),
        "mean_protected_matches": (
            float(np.mean(acc["protected_counts"]))
            if acc["protected_counts"] else 0.0
        ),
        "positive_retained_queries": acc[
            "positive_retained_queries"
        ],
        "candidate_recall": (
            acc["positive_retained_queries"] / query_count
            if query_count else np.nan
        ),
        "generated_queries": acc["generated_queries"],
        "candidate_rows": acc["candidate_rows"],
    })

generation_df = pd.DataFrame(generation_rows)

generation_df.to_csv(SITE_REPORT_CSV, index=False)
query_report_df.to_csv(QUERY_REPORT_CSV, index=False)
query_report_df.to_csv(QUERY_MANIFEST_CSV, index=False)
issues_df.to_csv(ISSUES_CSV, index=False)
missing_positive_df.to_csv(
    MISSING_POSITIVE_AUDIT_CSV, index=False
)

display(generation_df)
print("Working pair CSV           :", WORKING_PAIR_CSV)
print("Query manifest             :", QUERY_MANIFEST_CSV)
print("Query report               :", QUERY_REPORT_CSV)
print("Site report                :", SITE_REPORT_CSV)
print("Issues                     :", ISSUES_CSV)
print("Missing-positive diagnostic:", MISSING_POSITIVE_AUDIT_CSV)
print("Feature audit               :", FEATURE_AUDIT_CSV)


Prepare sites:   0%|          | 0/22 [00:00<?, ?it/s]

amazon-2017-2020:   0%|          | 0/76 [00:00<?, ?it/s]

apple-2018-2020:   0%|          | 0/62 [00:00<?, ?it/s]

beijing-2017-2019:   0%|          | 0/103 [00:00<?, ?it/s]

Bing:   0%|          | 0/11 [00:00<?, ?it/s]

...

youtube:   0%|          | 0/70 [00:00<?, ?it/s]

Prepared exact query plans: 1100
Manual shortlist top-K: 25


Write shortlisted pairs:   0%|          | 0/1100 [00:00<?, ?it/s]

,site,split,new_dom_nodes,old_geometry_nodes,new_geometry_nodes,old_visible_nodes,new_visible_nodes,old_enabled_nodes,new_enabled_nodes,resolved_parent_links,...,mean_shortlist_candidates,median_shortlist_candidates,max_shortlist_candidates,mean_shortlist_dom_fraction,mean_fallback_selected,mean_protected_matches,positive_retained_queries,candidate_recall,generated_queries,candidate_rows
0,amazon-2017-2020,development,561,252,561,252,561,0,0,560,...,25.013158,25.0,26,0.044587,25.000000,0.947368,76,1.000000,76,1901
1,apple-2018-2020,development,211,140,211,140,211,0,0,210,...,25.000000,25.0,25,0.118483,24.887097,1.209677,62,1.000000,62,1550
2,beijing-2017-2019,test,890,729,890,729,890,0,0,889,...,25.000000,25.0,25,0.028090,19.213592,1.601942,103,1.000000,103,2575
3,Bing,development,41,70,41,70,41,0,0,40,...,25.000000,25.0,25,0.609756,12.545455,0.909091,11,1.000000,11,275
4,book-2016-2019,test,539,597,539,597,539,0,0,538,...,25.018692,25.0,26,0.046417,24.009346,1.056075,107,1.000000,107,2677
5,facebook-2020-2022,development,61,121,61,121,61,0,0,60,...,25.022727,25.0,26,0.410209,5.659091,1.000000,44,1.000000,44,1101
6,fandom,development,432,426,432,426,432,0,0,431,...,25.012048,25.0,26,0.057898,24.939759,1.771084,83,1.000000,83,2076
7,foodnetwork-2018-2019,development,351,425,351,425,351,0,0,350,...,25.000000,25.0,25,0.071225,17.111111,0.629630,27,1.000000,27,675
8,google,development,37,36,37,36,37,0,0,36,...,24.181818,25.0,25,0.653563,9.727273,1.000000,10,0.909091,10,241
9,instagram,test,59,58,59,58,59,0,0,58,...,25.000000,25.0,25,0.423729,12.500000,0.833333,12,1.000000,12,300


Working pair CSV           : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/pairs_real_query_shortlist_presemantic.csv
Query manifest             : /workspaces/REPATH-Evaluation-Suite/artifacts/repath/repath_query_manifest.csv
Query report               : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_query_shortlist_query_report.csv
Site report                : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_query_shortlist_site_report.csv
Issues                     : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_query_shortlist_issues.csv
Missing-positive diagnostic: /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_query_shortlist_missing_positive_diagnostics.csv
Feature audit               : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_query_shortlist_feature_audit.csv


## 6. Compute semantic features and export the 50-feature dataset

The candidate rows already contain the raw text and context strings that the runtime implementation will receive from stored failed-element metadata and live Selenium candidates. This cell converts those strings into the five SBERT similarities and writes the final CSV containing all 50 numerical model features.


In [7]:
SEMANTIC_TEXT_PAIRS = {
    "semantic_sim": ("old_text_n", "new_text_n"),
    "parent_text_sim": ("old_parent_text_n", "new_parent_text_n"),
    "sibling_text_sim": ("old_sibling_text_n", "new_sibling_text_n"),
    "child_text_sim": ("old_child_text_n", "new_child_text_n"),
    "context_text_sim": ("old_context_text_n", "new_context_text_n"),
}

raw_pairs = pd.read_csv(WORKING_PAIR_CSV, keep_default_na=False)

required_raw_columns = {
    column
    for pair in SEMANTIC_TEXT_PAIRS.values()
    for column in pair
}
missing_raw_columns = sorted(required_raw_columns - set(raw_pairs.columns))
if missing_raw_columns:
    raise ValueError(
        f"Working pair CSV is missing semantic text columns: "
        f"{missing_raw_columns}"
    )

sbert = SentenceTransformer(SBERT_MODEL_NAME)

if SBERT_CACHE.exists():
    text_to_emb = joblib.load(SBERT_CACHE)
    print(f"Loaded SBERT cache: {len(text_to_emb):,} entries")
else:
    empty_embedding = sbert.encode(
        [""],
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    text_to_emb = {"": np.asarray(empty_embedding, dtype=np.float32)}
    print("Started a fresh SBERT cache")


def cache_texts(texts, batch_size=SBERT_BATCH_SIZE):
    normalized = [normalise_text(value) for value in texts]
    missing = [
        value
        for value in dict.fromkeys(normalized)
        if value and value not in text_to_emb
    ]
    for start in tqdm(
        range(0, len(missing), batch_size),
        desc="Encode unique semantic texts",
    ):
        batch = missing[start:start + batch_size]
        embeddings = sbert.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        for value, embedding in zip(batch, embeddings):
            text_to_emb[value] = np.asarray(embedding, dtype=np.float32)


def batch_cosine_similarity(left, right):
    left_values = left.fillna("").astype(str).map(normalise_text)
    right_values = right.fillna("").astype(str).map(normalise_text)

    zero = text_to_emb[""]
    left_matrix = np.stack([
        text_to_emb.get(value, zero)
        for value in left_values
    ])
    right_matrix = np.stack([
        text_to_emb.get(value, zero)
        for value in right_values
    ])

    return np.sum(
        left_matrix * right_matrix,
        axis=1,
    ).astype(np.float32)


all_semantic_texts = []
for old_column, new_column in SEMANTIC_TEXT_PAIRS.values():
    all_semantic_texts.extend(
        raw_pairs[old_column].fillna("").astype(str).tolist()
    )
    all_semantic_texts.extend(
        raw_pairs[new_column].fillna("").astype(str).tolist()
    )

cache_texts(all_semantic_texts)

for feature, (old_column, new_column) in SEMANTIC_TEXT_PAIRS.items():
    raw_pairs[feature] = batch_cosine_similarity(
        raw_pairs[old_column],
        raw_pairs[new_column],
    )

missing_model_features = sorted(
    set(ALL_MODEL_FEATURES) - set(raw_pairs.columns)
)
if missing_model_features:
    raise ValueError(
        f"Final pair CSV is missing model features: "
        f"{missing_model_features}"
    )

for feature in ALL_MODEL_FEATURES:
    raw_pairs[feature] = pd.to_numeric(
        raw_pairs[feature],
        errors="raise",
    ).astype(np.float32)

feature_matrix = raw_pairs[ALL_MODEL_FEATURES].to_numpy(dtype=np.float32)
if not np.isfinite(feature_matrix).all():
    raise ValueError("The final 50-feature matrix contains NaN or infinity")

raw_pairs.to_csv(PAIR_CSV, index=False)
joblib.dump(text_to_emb, SBERT_CACHE)

feature_group_map = {
    **{feature: "lexical" for feature in MODEL_LEXICAL_FEATURES},
    **{feature: "semantic" for feature in MODEL_SEMANTIC_FEATURES},
    **{feature: "structural" for feature in MODEL_STRUCTURAL_FEATURES},
    **{feature: "attribute" for feature in MODEL_ATTRIBUTE_FEATURES},
    **{feature: "geometry" for feature in MODEL_GEOMETRY_FEATURES},
}

feature_schema_df = pd.DataFrame({
    "vector_index": np.arange(len(ALL_MODEL_FEATURES), dtype=int),
    "feature": ALL_MODEL_FEATURES,
    "group": [feature_group_map[feature] for feature in ALL_MODEL_FEATURES],
    "dtype": "float32",
    "semantic_model": [
        SBERT_MODEL_NAME if feature in MODEL_SEMANTIC_FEATURES else ""
        for feature in ALL_MODEL_FEATURES
    ],
})
feature_schema_df.to_csv(FEATURE_SCHEMA_CSV, index=False)

if not KEEP_INTERMEDIATE_PAIR_CSV and WORKING_PAIR_CSV.exists():
    WORKING_PAIR_CSV.unlink()

print("Final model-ready pair CSV:", PAIR_CSV)
print("Query manifest CSV         :", QUERY_MANIFEST_CSV)
print("Feature schema CSV         :", FEATURE_SCHEMA_CSV)
print("SBERT cache                :", SBERT_CACHE)
print("Rows                       :", f"{len(raw_pairs):,}")
print("Model features             :", len(ALL_MODEL_FEATURES))
display(feature_schema_df)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Started a fresh SBERT cache


Encode unique semantic texts:   0%|          | 0/82 [00:00<?, ?it/s]

Final model-ready pair CSV: /workspaces/REPATH-Evaluation-Suite/generated/repath/features/pairs_real_query_shortlist_50_features.csv
Query manifest CSV         : /workspaces/REPATH-Evaluation-Suite/artifacts/repath/repath_query_manifest.csv
Feature schema CSV         : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_50_feature_schema.csv
SBERT cache                : /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_sbert_embedding_cache.pkl
Rows                       : 27,420
Model features             : 50


,vector_index,feature,group,dtype,semantic_model
0,0,idx_distance_log,lexical,float32,
1,1,old_text_len,lexical,float32,
2,2,new_text_len,lexical,float32,
3,3,len_diff,lexical,float32,
4,4,text_len_ratio,lexical,float32,
5,5,text_exact,lexical,float32,
6,6,text_substr,lexical,float32,
7,7,old_text_empty,lexical,float32,
8,8,new_text_empty,lexical,float32,
9,9,text_token_overlap,lexical,float32,


## 7. Audit missing positive candidates

A positive may be absent because it never entered the balanced pool or because its cheap retrieval rank fell below `QUERY_SHORTLIST_TOP_K`. Exact matches are protected without consulting the ground-truth label.


In [8]:
if missing_positive_df.empty:
    print("No missing positives: every framework target was retained.")
else:
    reason_by_site = (
        missing_positive_df
        .groupby(["site", "reject_reason"])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )

    reason_overall = (
        missing_positive_df["reject_reason"]
        .value_counts()
        .rename_axis("reject_reason")
        .reset_index(name="missing_queries")
    )
    reason_overall["percentage"] = (
        reason_overall["missing_queries"]
        / reason_overall["missing_queries"].sum()
        * 100
    ).round(2)

    print("Overall missing-positive reasons")
    display(reason_overall)

    print("Missing-positive counts by site")
    display(reason_by_site)

    diagnostic_columns = [
        "qid", "site", "split", "old_idx", "true_new_idx",
        "reject_reason", "full_balanced_candidate_count",
        "candidate_count", "shortlist_top_k",
        "shortlist_protected_count",
        "positive_in_full_balanced_pool",
        "positive_shortlist_rank", "positive_shortlist_score",
        "positive_shortlist_protected",
        "positive_shortlist_protection_reason",
        "positive_candidate_source", "positive_tag",
        "positive_role", "positive_text",
        "positive_accessible_text",
        "positive_accessible_text_length",
        "positive_displayed", "positive_width",
        "positive_height", "interactive_ok",
        "text_leaf_ok", "interactive_descendant_ok",
        "base_candidate_ok", "fallback_selected",
        "fallback_rank", "fallback_score",
    ]
    diagnostic_columns = [
        column for column in diagnostic_columns
        if column in missing_positive_df.columns
    ]

    print("Individual missing positives")
    display(
        missing_positive_df[diagnostic_columns]
        .sort_values(["site", "reject_reason", "qid"])
        .reset_index(drop=True)
    )

print("Saved diagnostic CSV:", MISSING_POSITIVE_AUDIT_CSV)


Overall missing-positive reasons


,reject_reason,missing_queries,percentage
0,outside_query_shortlist,4,80.0
1,fallback_score_below_threshold,1,20.0


Missing-positive counts by site


reject_reason,fallback_score_below_threshold,outside_query_shortlist
site,,
google,0,1
movie-2014-2019,1,0
xfinity-2018-2020,0,3


Individual missing positives


,qid,site,split,old_idx,true_new_idx,reject_reason,full_balanced_candidate_count,candidate_count,shortlist_top_k,shortlist_protected_count,...,positive_displayed,positive_width,positive_height,interactive_ok,text_leaf_ok,interactive_descendant_ok,base_candidate_ok,fallback_selected,fallback_rank,fallback_score
0,google::4,google,development,4,16,outside_query_shortlist,30,25,25,1,...,True,33.0,44.0,False,False,False,False,True,9.0,0.17819
1,movie-2014-2019::201,movie-2014-2019,test,201,240,fallback_score_below_threshold,162,25,25,0,...,True,90.0,14.0,False,False,False,False,False,57.0,0.14483
2,xfinity-2018-2020::6,xfinity-2018-2020,test,6,18,outside_query_shortlist,140,25,25,0,...,True,71.0,26.0,True,False,False,True,False,NaN,NaN
3,xfinity-2018-2020::7,xfinity-2018-2020,test,7,7,outside_query_shortlist,140,25,25,0,...,True,48.0,20.0,True,False,False,True,False,NaN,NaN
4,xfinity-2018-2020::87,xfinity-2018-2020,test,87,164,outside_query_shortlist,140,25,25,0,...,True,33.0,27.0,True,False,False,True,False,NaN,NaN


Saved diagnostic CSV: /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_query_shortlist_missing_positive_diagnostics.csv


## 8. Verify generated files


In [9]:
expected_qids = set(queries_df["qid"].astype(str))
written_qids = set()
positive_counts = Counter()
row_counts = Counter()
declared_candidate_counts = {}

for chunk in pd.read_csv(
    PAIR_CSV,
    chunksize=200_000,
    keep_default_na=False,
):
    for row in chunk.itertuples(index=False):
        qid = str(row.qid)
        written_qids.add(qid)
        positive_counts[qid] += int(row.label)
        row_counts[qid] += 1
        declared_candidate_counts.setdefault(
            qid, int(row.n_candidates)
        )

bad_positive_qids = [
    qid for qid, count in positive_counts.items()
    if count != 1
]
bad_candidate_count_qids = [
    qid for qid, count in row_counts.items()
    if declared_candidate_counts[qid] != count
]

retained_qids = set(
    query_report_df.loc[
        query_report_df["positive_retained"], "qid"
    ].astype(str)
)

total_pair_rows = int(sum(row_counts.values()))
mean_rows_per_written_query = (
    total_pair_rows / len(written_qids)
    if written_qids else 0.0
)

print("Manual shortlist top-K           :", shortlist_label(selected_top_k))
print("Exact framework queries          :", len(expected_qids))
print("Positive retained                :", len(retained_qids))
print(
    "Candidate recall                 :",
    f"{len(retained_qids) / len(expected_qids):.2%}",
)
print("Queries written to pair CSV      :", len(written_qids))
print("Pair rows                        :", total_pair_rows)
print(
    "Mean rows per written query      :",
    f"{mean_rows_per_written_query:.2f}",
)
print(
    "Positive-missing queries         :",
    len(expected_qids - retained_qids),
)
print("Bad positive groups              :", len(bad_positive_qids))
print(
    "Bad n_candidates groups          :",
    len(bad_candidate_count_qids),
)

assert not (written_qids - expected_qids)
assert not bad_positive_qids
assert not bad_candidate_count_qids

query_report_df["written_candidate_rows"] = np.where(
    query_report_df["written_to_pair_csv"],
    query_report_df["candidate_count"],
    0,
)

split_summary = (
    query_report_df.groupby("split", as_index=False)
    .agg(
        framework_queries=("qid", "nunique"),
        positive_retained=("positive_retained", "sum"),
        mean_full_balanced_candidates=(
            "full_balanced_candidate_count", "mean"
        ),
        mean_shortlist_candidates=("candidate_count", "mean"),
        median_shortlist_candidates=("candidate_count", "median"),
        mean_candidate_dom_fraction=(
            "candidate_dom_fraction", "mean"
        ),
        mean_protected_matches=(
            "shortlist_protected_count", "mean"
        ),
        pair_rows=("written_candidate_rows", "sum"),
    )
)
split_summary["candidate_recall"] = (
    split_summary["positive_retained"]
    / split_summary["framework_queries"]
)

print("\nResults for the manually configured shortlist")
display(split_summary)

print("\nPer-site results")
display(generation_df[[
    "split", "site", "expected_framework_queries",
    "new_dom_nodes", "mean_full_balanced_candidates",
    "mean_shortlist_candidates",
    "mean_shortlist_dom_fraction",
    "mean_protected_matches",
    "positive_retained_queries", "candidate_recall",
    "candidate_rows",
]])

overall_mean_dom_fraction = float(
    query_report_df["candidate_dom_fraction"].mean()
)
print(
    "\nMean fraction of each new DOM retained:",
    f"{overall_mean_dom_fraction:.2%}",
)

if not missing_positive_df.empty:
    print("\nMissing positives by split and reason")
    display(
        missing_positive_df
        .groupby(["split", "reject_reason"], as_index=False)
        .agg(missing_queries=("qid", "nunique"))
        .sort_values(
            ["split", "missing_queries"],
            ascending=[True, False],
        )
    )

print("\nPositive source distribution")
display(
    query_report_df.loc[
        query_report_df["positive_retained"]
    ]
    .groupby(
        ["split", "positive_candidate_source"],
        as_index=False,
    )
    .agg(queries=("qid", "nunique"))
    .sort_values(
        ["split", "queries"],
        ascending=[True, False],
    )
)


# Deterministic-feature completeness and zero/constant audit
audit_columns = ALL_MODEL_FEATURES + ["geometry_available"]
audit_state = {
    column: {
        "rows": 0,
        "non_null": 0,
        "non_zero": 0,
        "min": np.inf,
        "max": -np.inf,
        "values": set(),
    }
    for column in audit_columns
}

geometry_split_rows = []

for chunk in pd.read_csv(
    PAIR_CSV,
    chunksize=200_000,
    keep_default_na=False,
):
    missing_columns = [
        column for column in audit_columns
        if column not in chunk.columns
    ]
    if missing_columns:
        raise AssertionError(
            f"Generated pair CSV is missing columns: {missing_columns}"
        )

    geometry_split_rows.append(
        chunk.groupby("split", as_index=False).agg(
            rows=("qid", "size"),
            geometry_available=("geometry_available", "sum"),
        )
    )

    for column in audit_columns:
        numeric = pd.to_numeric(chunk[column], errors="coerce")
        finite = numeric[np.isfinite(numeric)]
        state = audit_state[column]
        state["rows"] += len(numeric)
        state["non_null"] += int(finite.notna().sum())
        state["non_zero"] += int((finite != 0).sum())
        if not finite.empty:
            state["min"] = min(state["min"], float(finite.min()))
            state["max"] = max(state["max"], float(finite.max()))
            if len(state["values"]) <= 25:
                state["values"].update(
                    float(value)
                    for value in finite.drop_duplicates().head(26)
                )

feature_audit_rows = []
for column, state in audit_state.items():
    minimum = state["min"] if np.isfinite(state["min"]) else np.nan
    maximum = state["max"] if np.isfinite(state["max"]) else np.nan
    feature_audit_rows.append({
        "feature": column,
        "rows": state["rows"],
        "non_null_rows": state["non_null"],
        "non_zero_rows": state["non_zero"],
        "non_zero_fraction": (
            state["non_zero"] / state["rows"]
            if state["rows"] else np.nan
        ),
        "min": minimum,
        "max": maximum,
        "observed_unique_values_at_least": len(state["values"]),
        "all_zero": bool(state["rows"] and state["non_zero"] == 0),
        "constant": bool(
            state["rows"]
            and len(state["values"]) <= 1
        ),
    })

feature_audit_df = pd.DataFrame(feature_audit_rows)
feature_audit_df.to_csv(FEATURE_AUDIT_CSV, index=False)

print("\nComplete 50-feature audit")
display(feature_audit_df)

all_zero_features = feature_audit_df.loc[
    feature_audit_df["all_zero"], "feature"
].tolist()
print("All-zero generated columns:", all_zero_features)

if geometry_split_rows:
    geometry_by_split = (
        pd.concat(geometry_split_rows, ignore_index=True)
        .groupby("split", as_index=False)
        .sum(numeric_only=True)
    )
    geometry_by_split["geometry_fraction"] = (
        geometry_by_split["geometry_available"]
        / geometry_by_split["rows"]
    )
    print("\nGeometry availability by split")
    display(geometry_by_split)

print("Saved complete feature audit:", FEATURE_AUDIT_CSV)


Manual shortlist top-K           : 25
Exact framework queries          : 1100
Positive retained                : 1095
Candidate recall                 : 99.55%
Queries written to pair CSV      : 1095
Pair rows                        : 27420
Mean rows per written query      : 25.04
Positive-missing queries         : 5
Bad positive groups              : 0
Bad n_candidates groups          : 0

Results for the manually configured shortlist


,split,framework_queries,positive_retained,mean_full_balanced_candidates,mean_shortlist_candidates,median_shortlist_candidates,mean_candidate_dom_fraction,mean_protected_matches,pair_rows,candidate_recall
0,development,782,781,228.604859,25.054987,25.0,0.146016,1.483376,19568,0.998721
1,test,318,314,371.993711,25.006289,25.0,0.075608,1.116352,7852,0.987421



Per-site results


,split,site,expected_framework_queries,new_dom_nodes,mean_full_balanced_candidates,mean_shortlist_candidates,mean_shortlist_dom_fraction,mean_protected_matches,positive_retained_queries,candidate_recall,candidate_rows
0,development,amazon-2017-2020,76,561,332.000000,25.013158,0.044587,0.947368,76,1.000000,1901
1,development,apple-2018-2020,62,211,158.887097,25.000000,0.118483,1.209677,62,1.000000,1550
2,test,beijing-2017-2019,103,890,630.213592,25.000000,0.028090,1.601942,103,1.000000,2575
3,development,Bing,11,41,31.545455,25.000000,0.609756,0.909091,11,1.000000,275
4,test,book-2016-2019,107,539,344.009346,25.018692,0.046417,1.056075,107,1.000000,2677
5,development,facebook-2020-2022,44,61,54.659091,25.022727,0.410209,1.000000,44,1.000000,1101
6,development,fandom,83,432,318.939759,25.012048,0.057898,1.771084,83,1.000000,2076
7,development,foodnetwork-2018-2019,27,351,236.111111,25.000000,0.071225,0.629630,27,1.000000,675
8,development,google,11,37,28.727273,24.181818,0.653563,1.000000,10,0.909091,241
9,test,instagram,12,59,44.500000,25.000000,0.423729,0.833333,12,1.000000,300



Mean fraction of each new DOM retained: 12.57%

Missing positives by split and reason


...


Complete 50-feature audit


,feature,rows,non_null_rows,non_zero_rows,non_zero_fraction,min,max,observed_unique_values_at_least,all_zero,constant
0,idx_distance_log,27420,27420,26947,0.982750,0.000000,6.782192,26,False,False
1,old_text_len,27420,27420,25195,0.918855,0.000000,788.000000,26,False,False
2,new_text_len,27420,27420,25168,0.917870,0.000000,2965.000000,26,False,False
3,len_diff,27420,27420,20572,0.750255,0.000000,2918.000000,26,False,False
4,text_len_ratio,27420,27420,24789,0.904048,0.000000,1.000000,26,False,False
5,text_exact,27420,27420,1177,0.042925,0.000000,1.000000,2,False,False
6,text_substr,27420,27420,2263,0.082531,0.000000,1.000000,2,False,False
7,old_text_empty,27420,27420,2225,0.081145,0.000000,1.000000,2,False,False
8,new_text_empty,27420,27420,2252,0.082130,0.000000,1.000000,2,False,False
9,text_token_overlap,27420,27420,4662,0.170022,0.000000,1.000000,2,False,False


All-zero generated columns: []

Geometry availability by split


,split,rows,geometry_available,geometry_fraction
0,development,19568,19568,1.0
1,test,7852,7852,1.0


Saved complete feature audit: /workspaces/REPATH-Evaluation-Suite/generated/repath/features/repath_query_shortlist_feature_audit.csv


## Notes

The notebook uses only the manually specified `QUERY_SHORTLIST_TOP_K`; it
does not tune the shortlist using cross-validation or held-out test results.

The frozen site-level partition is:

- development: 16 websites;
- held-out test: beijing-2017-2019, book-2016-2019, instagram, movie-2014-2019, xfinity-2018-2020 and yahoo.

The training notebook creates four deterministic, site-disjoint folds from
the 16 development websites. Each fold contains four websites. Hyperparameter
selection, early-stopping round selection, and confidence-threshold
calibration use development websites only. The six test websites remain
outside every cross-validation fold.

The exported pair CSV contains the complete fixed **50-feature** vector:

- 12 lexical features;
- 5 semantic SBERT features;
- 8 structural features;
- 20 attribute features;
- 5 geometry features.

The structural group contains only DOM-tree and XPath relationships:

```text
tag_match
dom_depth_norm_diff
parent_chain_similarity
ancestor_path_similarity
xpath_token_sim
sibling_position_diff
sibling_context_jaccard
child_tag_jaccard
```

The exact UITestFix match has relevance 3 and every incorrect candidate has
relevance 0. Every candidate row receives equal treatment during model
training.

The training notebook consumes only:

1. `pairs_real_query_shortlist_50_features.csv`;
2. `repath_query_manifest.csv`;
3. `repath_50_feature_schema.csv`.

It does not access UITestFix JSON files or recompute any feature.


## Reference output

The bundled model artifacts were produced from the following retrieval output:

| Split | Framework queries | Positive retained | Candidate recall | Mean shortlist | Pair rows |
| --- | ---: | ---: | ---: | ---: | ---: |
| Development | 782 | 781 | 99.87% | 25.05 | 19,568 |
| Held-out test | 318 | 314 | 98.74% | 25.01 | 7,852 |

The generated feature matrices remain under `generated/repath/features/` and are not committed. The manifest under `artifacts/repath/` is consumed by the training and calibration stages.
